# Mechanism study E0-E8: why smooth transport decodes well (KEEP, weak)

Its only paper number (38% retained variance, `TargetVariance`) is also produced by `paper_completion.ipynb`. Kept because it renders control directories (`tm_scale1000/{target_mean,gauss_full,meanshift*,...}`) that `presubmission_additions.ipynb` reads.

**Needs:** DTD, the CNT checkpoint.

**Maintained runnable path:** none dedicated; `score_only.py` rescores `TargetVariance` from `paper_completion.ipynb`'s cached output instead.

# Mechanism study — why smooth transport decodes well (E0–E7, minus human study)

Implements the plan in `OT_Mechanism_Theory_and_Experiment_Plan.md` on the real CNT pipeline, in order of importance:

| Cell | Experiment | What it decides | Time (T4/A100) |
|---|---|---|---|
| 0 | Setup | canonical protocol, CNT, observers, helpers | ~5 min |
| 1 | **E0 audit** (H6) | is the intervention real? every SPADE stage consumes the transported map; Sinkhorn convergence | ~2 min |
| 2 | **E2 subspace geometry** (H1) | per matching rule: off-subspace energy `e_sub`, affine residual `e_aff`, commutation error; does `e_sub` predict the rules' scores? | ~10 min |
| 3 | **E1 ε×t grid** (H2) | is ε-robustness specific to t≈0.5? endpoint effects | ~25 min |
| 4 | **E3 causal projection** (H1) | remove/add the off-subspace component of the hard map with matched controls | ~30 min |
| 5 | **E4 amplitude / extension / budget** (H4) | is the hard–soft gap smoothness, residual amplitude, or the NN extension? | ~25 min |
| 6 | **E6 target information** | what barycentric projection discards: conditional covariance, motif coverage, rare clusters | ~15 min |
| 7 | **E5 decoder sensitivity** (H3) | which perturbation directions the decoder actually responds to; per-stage swaps | ~15 min |
| 8 | **E7 confirmation** | frozen λ* projected transport vs baselines on the frozen 120 pairs, KID with paired pair-resampling | ~20 min |
| 9 | Summary tables + figures | everything into `mechanism/summary.json` and figures | ~2 min |

Every cell is resumable (results cached to Drive under `DATA_ROOT/mechanism/`); "Run all" is safe after a disconnect.
Development pairs (24) are pair-disjoint from the 120 confirmation pairs (`pairs_120`). No paid generation anywhere.
Human-study endpoints are replaced by: DINO/Inception manifold realism, round-trip discrepancy, target motif coverage, endpoint proximity, and LPIPS — stated as such.

In [ ]:
# ============================================================
# 0. SETUP — canonical protocol (identical to finish_day cell 0) + mechanism helpers
# ============================================================
import os,sys,glob,random,time,gc,math,json,zipfile,hashlib,itertools
import numpy as np, pandas as pd, torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageFile
from scipy.stats import wilcoxon, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy import linalg
from google.colab import drive
drive.mount('/content/drive')
ImageFile.LOAD_TRUNCATED_IMAGES=True; device='cuda'
assert torch.cuda.is_available(),'switch runtime to GPU'
BASE=os.environ.get('DATA_ROOT', '/content/data')  # was hardcoded to the Drive mount; set DATA_ROOT to point at your local copy
SCALE=f'{BASE}/tm_scale1000'; OUT=f'{BASE}/matching_ablation'
MECH=f'{BASE}/mechanism'; os.makedirs(MECH,exist_ok=True); FIGS=f'{MECH}/figs'; os.makedirs(FIGS,exist_ok=True)
TMP='/content/tmp_imgs'; os.makedirs(TMP,exist_ok=True)
t0=time.time()

# --- DTD ---
LOCAL='/content/dtd_images'; ZIP=f'{BASE}/dtd_images.zip'
if len(glob.glob(f'{LOCAL}/*/*.jpg'))<5000:
    os.system('rm -rf /content/_dtd'); os.system(f'unzip -qo "{ZIP}" -d /content/_dtd')
    inner=os.listdir('/content/_dtd'); src='/content/_dtd/images' if 'images' in inner else f'/content/_dtd/{inner[0]}'
    os.system(f'rm -rf {LOCAL} && mv "{src}" {LOCAL}')
cats=sorted(os.listdir(LOCAL)); print(f'DTD ready: {len(cats)} cats, {time.time()-t0:.0f}s')
os.system('pip install -q timm lpips einops 2>/dev/null')

# --- CNT ---
os.chdir('/content'); os.system('rm -rf cnt-siga24; git clone -q https://github.com/phtu-cs/compositional-neural-textures.git cnt-siga24')
os.system(f'rm -rf /content/cnt-siga24/ckpts; ln -sfn "{BASE}/cnt_ckpts" /content/cnt-siga24/ckpts')
CNT='/content/cnt-siga24'; os.chdir(CNT); sys.path.insert(0,CNT)
try:
    from src.inference.core import TexAutoEncoderInference, load_image
except Exception as e:
    print('installing missing deps:',type(e).__name__)
    os.system('pip install -q -r requirements.in; pip install -q "git+https://github.com/facebookresearch/detectron2.git"')
    from src.inference.core import TexAutoEncoderInference, load_image
model=TexAutoEncoderInference(ckpt_path=Path(f'{CNT}/ckpts/finetuned-model256.ckpt'),device=device)
IMG_SIZE=256; FMS=[round(IMG_SIZE*r) for r in model.generator.feature_map_size_ratios]
EPS,ITERS,NSUB=0.05,100,4096
CNT_COMMIT=os.popen(f'cd {CNT} && git rev-parse HEAD').read().strip()
print(f'model ready {time.time()-t0:.0f}s | FMS={FMS} | CNT commit {CNT_COMMIT[:10]}')

_tc=[0]
def cnt_load(x,size=IMG_SIZE):
    if isinstance(x,(str,Path)): return load_image(Path(x),size).to(device)
    _tc[0]+=1; fp=f'{TMP}/t{_tc[0]%64}.png'; x.convert('RGB').save(fp); return load_image(Path(fp),size).to(device)
def to_pil(dec):
    a=((dec.squeeze(0).detach().cpu().clamp(-1,1)+1)/2).permute(1,2,0).numpy(); return Image.fromarray((a*255).astype(np.uint8))
def get_gen_inputs(img):
    with torch.no_grad():
        blob=model.encode(img); L=model.splat(blob)
        return blob,L["layer_level_layouts"][-1][0]["blob_features"].clone(),L["layer_level_layouts"][-1][0]["score_img"].clone()
def _pyr(si):
    sp={}; s=si; sp[s.shape[-1]]=s
    while s.shape[-1]>min(FMS): s=F.interpolate(s,s.shape[-1]//2,mode="bilinear",align_corners=False); sp[s.shape[-1]]=s
    return sp
def build_style_maps(bf,si,fms):
    sp=_pyr(si); return {size: torch.einsum("bnc,bnhw->bchw",bf,sp[size]) for size in set(fms)}
def decode_with_style_maps(blob_ref,sm):
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm[fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def decode_stagewise(blob_ref,sm_by_stage):
    """same decoder, but stage i receives sm_by_stage[i] (dict size->map). For E5 per-stage swaps."""
    with torch.no_grad():
        L=model.splat(blob_ref); si=L["layer_level_layouts"][-1][0]["score_img"]
        fms=[round(si.shape[-1]*r) for r in model.generator.feature_map_size_ratios]; sp=_pyr(si)
        fi=sp[fms[0]][:,1:].amax(dim=1)[:,None].repeat(1,model.generator.n_embedding,1,1); x=fi.clone()
        for i in range(len(fms)):
            x=model.generator.spade_blocks[i](x,sm_by_stage[i][fms[i]])
            if i==len(fms)-1: break
            elif fms[i]!=fms[i+1]: x=F.interpolate(x,size=x.shape[-1]*2,mode="bilinear",align_corners=False)
        return torch.tanh(x.unsqueeze(1)).flatten(1,2)
def build_d(ia,ib):
    ba,bfa,sia=get_gen_inputs(ia); bb,bfb,sib=get_gen_inputs(ib)
    return {"blob_a":ba,"bf_a":bfa,"si_a":sia,"blob_b":bb,"bf_b":bfb,"si_b":sib}
def _grids(d):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    L=max(FMS); A,B=sa[L][0],sb[L][0]; Cc,H,W=A.shape
    return L,Cc,H,W,A.permute(1,2,0).reshape(-1,Cc),B.permute(1,2,0).reshape(-1,Cc)
_TAG={'s':1,'t':2}
def _sub(N,seed,tag):
    g=torch.Generator(device=device).manual_seed(int(seed)*7919+_TAG[tag]); return torch.randperm(N,generator=g,device=device)[:NSUB]
def sm_from_field(Z,L,H,W,Cc):
    it=Z.reshape(H,W,Cc).permute(2,0,1).unsqueeze(0)
    return {s:(it if s==L else F.interpolate(it,size=s,mode="bilinear",align_corners=False)) for s in set(FMS)}
def _dec(d,Z,L,H,W,Cc,layout='a'): return decode_with_style_maps(d["blob_"+layout],sm_from_field(Z.float(),L,H,W,Cc))
def interp_lin_pixel(d,eta):
    sa=build_style_maps(d["bf_a"],d["si_a"],FMS); sb=build_style_maps(d["bf_b"],d["si_b"],FMS)
    return decode_with_style_maps(d["blob_a"],{s:(1-eta)*sa[s]+eta*sb[s] for s in set(FMS)})
def make_recon(pil):
    e=cnt_load(pil); return interp_lin_pixel(build_d(e,e),0.0)
def pair_imgs(k): return Image.open(f'{SCALE}/A/{k:04d}.png').convert('RGB'), Image.open(f'{SCALE}/B/{k:04d}.png').convert('RGB')

# --- canonical Sinkhorn (as used in the paper) + converged log-domain Sinkhorn with duals ---
def sinkhorn_plan(x,y,eps,iters=ITERS):
    Cm=torch.cdist(x,y).pow(2); K=torch.exp(-Cm/eps)
    u=torch.ones(len(x),device=x.device)/len(x); v=torch.ones(len(y),device=y.device)/len(y)
    a=torch.ones_like(u); b=torch.ones_like(v)
    for _ in range(iters): a=u/(K@b+1e-30); b=v/(K.T@a+1e-30)
    return a[:,None]*K*b[None,:], Cm
def sinkhorn_log(x,y,eps,iters=3000,tol=1e-6):
    """balanced, uniform marginals, float64 log-domain; returns P, duals f,g, cost, marginal residual, iterations."""
    X=x.double(); Y=y.double(); Cm=torch.cdist(X,Y).pow(2); n,m=Cm.shape
    la=-math.log(n); lb=-math.log(m); f=torch.zeros(n,device=X.device,dtype=torch.float64); g=torch.zeros(m,device=X.device,dtype=torch.float64)
    for it in range(1,iters+1):
        f=eps*(la-torch.logsumexp((g[None,:]-Cm)/eps,1)); g=eps*(lb-torch.logsumexp((f[:,None]-Cm)/eps,0))
        if it%10==0:
            logP=(f[:,None]+g[None,:]-Cm)/eps; res=float((torch.exp(logP).sum(1)-1/n).abs().max()*n)
            if res<tol: break
    logP=(f[:,None]+g[None,:]-Cm)/eps; P=torch.exp(logP)
    res=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/m).abs().max()*m))
    return P,f,g,Cm,res,it
def plan_stats(P):
    P=P.double(); n=P.shape[0]; q=P/(P.sum(1,keepdim=True)+1e-300)
    Hrow=float(-(q*torch.log(q+1e-300)).sum(1).mean()); Htot=float(-(P*torch.log(P+1e-300)).sum())
    return dict(H_row=Hrow,H_total=Htot,MI_nats=math.log(n)-Hrow)
def natural_extension(x_all,y,g,eps,chunk=1024):
    """T(x)=sum_j q_j(x) y_j, q_j ∝ exp((g_j-||x-y_j||²)/eps) — dual-potential extension of a converged plan."""
    out=[]
    for i in range(0,len(x_all),chunk):
        Cm=torch.cdist(x_all[i:i+chunk].double(),y.double()).pow(2); q=torch.softmax((g[None,:]-Cm)/eps,1); out.append(q@y.double())
    return torch.cat(out)

# --- matching rules -> sampled targets (M,d); full field via canonical nn extension ---
def _targets(sf,tf,rule,seed,eps=EPS,M=NSUB,n_exact=1024,plan_kind='canonical'):
    seed=int(seed)   # numpy int64 from pandas breaks torch manual_seed
    N=sf.shape[0]; i_s=_sub(N,seed,'s')[:M]; i_t=_sub(N,seed,'t')[:M]; src,tgt=sf[i_s],tf[i_t]; info={}
    if rule in('ot','ot_sharp','ot_soft'):
        e={'ot':eps,'ot_sharp':0.005,'ot_soft':0.5}[rule] if rule!='ot' else eps
        if plan_kind=='canonical': P,_=sinkhorn_plan(src,tgt,e); info=dict(eps=e,**plan_stats(P))
        else: P,f,g,_,res,it=sinkhorn_log(src,tgt,e); info=dict(eps=e,res=res,iters=it,**plan_stats(P)); info['g']=g
        tt=((P/(P.sum(1,keepdim=True)+1e-30))@tgt.double()).float() if P.dtype==torch.float64 else (P/(P.sum(1,keepdim=True)+1e-30))@tgt
    elif rule=='exact':
        g_=torch.Generator(device=device).manual_seed(seed+11)
        js=torch.randperm(M,generator=g_,device=device)[:n_exact]; is_=torch.randperm(M,generator=g_,device=device)[:n_exact]
        r,c=linear_sum_assignment(torch.cdist(src[is_],tgt[js]).pow(2).cpu().numpy())
        tt=torch.full_like(src,float('nan')); tt[is_[torch.as_tensor(r,device=device)]]=tgt[js[torch.as_tensor(c,device=device)]]
        ok=~torch.isnan(tt[:,0]); tt[~ok]=tt[ok][torch.cdist(src[~ok],src[ok]).argmin(1)]; info=dict(n_exact=n_exact)
    elif rule=='nn': tt=tgt[torch.cdist(src,tgt).argmin(1)]
    elif rule=='gauss':
        X,Y=src.double(),tgt.double(); mx,my=X.mean(0),Y.mean(0); Xc=X-mx; Yc=Y-my; Cc=X.shape[1]; I=torch.eye(Cc,device=device,dtype=torch.float64); ridge=1e-4
        Sa=(Xc.T@Xc)/(len(X)-1)+ridge*I; Sb=(Yc.T@Yc)/(len(Y)-1)+ridge*I
        def ps(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.sqrt())@V.T
        def pis(M_,fl):
            e_,V=torch.linalg.eigh(.5*(M_+M_.T)); e_=e_.clamp_min(fl); return (V*e_.rsqrt())@V.T
        Sh=ps(Sa,ridge); Sih=pis(Sa,ridge); Amap=Sih@ps(Sh@Sb@Sh,0.0)@Sih; Amap=.5*(Amap+Amap.T)
        tt=(my+Xc@Amap.T).float(); info=dict(A=Amap,mu=(mx,my))
    elif rule=='uniform': tt=tgt.mean(0,keepdim=True).expand_as(src).clone()
    elif rule=='random':
        g_=torch.Generator(device=device).manual_seed(seed+7); tt=tgt[torch.randperm(M,generator=g_,device=device)]
    elif rule=='meanshift': tt=src+(tgt.mean(0)-src.mean(0))      # CNT-style native mean shift, adapted to the interpolation contract
    else: raise ValueError(rule)
    return i_s,tt,info
def field(d,rule,seed,**kw):
    seed=int(seed)
    """full-grid target field U (N,d) with the canonical nn extension; also returns X,Y grids and plan info."""
    L,Cc,H,W,sf,tf=_grids(d); i_s,tt,info=_targets(sf,tf,rule,seed,**kw)
    nn=torch.cdist(sf,sf[i_s]).argmin(1); return dict(U=tt[nn],X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,i_s=i_s,tt=tt,nn=nn,info=info)
def zt(fd,t): return (1-t)*fd['X']+t*fd['U']
def decode_field(d,fd,Z): return _dec(d,Z,fd['L'],fd['H'],fd['W'],fd['Cc'])

# --- fixed-layout subspace of A: X = W_A F_A ---
def splat_W(d,side='a'):
    sp=_pyr(d['si_'+side]); Wm=sp[max(FMS)][0]; K=Wm.shape[0]
    return Wm.reshape(K,-1).T.contiguous().double(), d['bf_'+side][0].double()      # (N,K),(K,d)
def projector(Wm,tol=1e-6):
    Uw,S,_=torch.linalg.svd(Wm,full_matrices=False); keep=S>S[0]*tol; return Uw[:,keep], int(keep.sum())
def proj(Q,U): U=U.double(); return Q@(Q.T@U)
def e_sub(Q,U):
    U=U.double(); Uc=U-U.mean(0,keepdim=True); off=U-proj(Q,U)
    return dict(e_sub=float(off.norm()/(Uc.norm()+1e-12)),off_rms=float(off.norm()/math.sqrt(U.numel())),cen_rms=float(Uc.norm()/math.sqrt(U.numel())))
def e_aff(X,U,seed=0):
    """affine fit U≈XA+b on half the rows, residual on held-out rows."""
    X=X.double(); U=U.double(); N=len(X); g=torch.Generator(device=X.device).manual_seed(seed); p=torch.randperm(N,generator=g,device=X.device); tr,te=p[:N//2],p[N//2:]
    Xa=torch.cat([X,torch.ones(N,1,device=X.device,dtype=torch.float64)],1)
    G=Xa[tr].T@Xa[tr]; sol=torch.linalg.solve(G+1e-6*torch.trace(G)/G.shape[0]*torch.eye(G.shape[0],device=G.device,dtype=G.dtype),Xa[tr].T@U[tr]); R=U[te]-Xa[te]@sol; Uc=U[te]-U[te].mean(0,keepdim=True)
    return dict(e_aff=float(R.norm()/(Uc.norm()+1e-12)),aff_rms=float(R.norm()/math.sqrt(R.numel())))
def commutation_err(Q,Wm,Fm,Tfun):
    """J = T(W F) - W T(F), for a frozen map T applied to blob vectors (out-of-support warning if far from cloud)."""
    X=(Wm@Fm); TX=Tfun(X.float()).double(); TF=Tfun(Fm.float()).double(); J=TX-Wm@TF
    return dict(comm=float(J.norm()/(TX-TX.mean(0)).norm()),comm_rms=float(J.norm()/math.sqrt(J.numel())))
def neighbor_disc(U,H,W):
    G=U.reshape(H,W,-1); return float(((G[:,1:]-G[:,:-1]).norm(dim=-1).mean()+(G[1:]-G[:-1]).norm(dim=-1).mean())/2)
def var_ret(U,Y): return float(U.double().var(0).sum()/Y.double().var(0).sum())
def interaction_scale(X,Y):
    """RMS of -2 Xc Yc^T (centered interaction) vs raw median cost: the calibration the plan asks for."""
    Xc=X.double()-X.double().mean(0); Yc=Y.double()-Y.double().mean(0); M_=-2*Xc@Yc.T
    return dict(inter_rms=float(M_.pow(2).mean().sqrt()),cost_med=float(torch.cdist(X.double(),Y.double()).pow(2).median()))

# --- observers (canonical) ---
import timm, lpips as lpips_lib
dino=timm.create_model('vit_small_patch14_dinov2.lvd142m',pretrained=True,num_classes=0).to(device).eval()
from torchvision.models import inception_v3, Inception_V3_Weights
_inc=inception_v3(weights=Inception_V3_Weights.IMAGENET1K_V1,aux_logits=True).to(device).eval(); _inc.fc=torch.nn.Identity()
for p in list(dino.parameters())+list(_inc.parameters()): p.requires_grad_(False)
_M=torch.tensor([0.485,0.456,0.406],device=device).view(1,3,1,1); _S=torch.tensor([0.229,0.224,0.225],device=device).view(1,3,1,1)
def _stack(pils): return torch.stack([torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/255. for p in pils]).to(device)
@torch.no_grad()
def dino_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=518,mode='bilinear',align_corners=False); out.append(dino((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
@torch.no_grad()
def inc_feats(pils,bs=64):
    out=[]
    for i in range(0,len(pils),bs):
        b=F.interpolate(_stack(pils[i:i+bs]),size=299,mode='bilinear',align_corners=False); out.append(_inc((b-_M)/_S).cpu().numpy())
    return np.concatenate(out)
lp=lpips_lib.LPIPS(net='alex').to(device).eval()
@torch.no_grad()
def lpips_d(p1,p2):
    t=lambda p:(torch.from_numpy(np.asarray(p.convert('RGB').resize((256,256))).copy()).float().permute(2,0,1)/127.5-1)[None].to(device)
    return float(lp(t(p1),t(p2)))
SCORE_RES=128
def at128(pil): return pil.convert('RGB').resize((SCORE_RES,SCORE_RES))
ref_inc=np.load(f'{OUT}/ref_inc.npy'); ref_dino=np.load(f'{OUT}/ref_dino.npy')
class Manifold:
    def __init__(self,R,k=3,n=1000,seed=0):
        rng=np.random.RandomState(seed); self.R=R[rng.choice(len(R),min(n,len(R)),replace=False)]
        D=np.linalg.norm(self.R[:,None]-self.R[None],axis=-1); D.sort(1); rad=D[:,k]
        keep=rad<=np.median(rad); self.R=self.R[keep]; self.rad=rad[keep]
    def __call__(self,X,bs=256):
        out=[]
        for i in range(0,len(X),bs):
            D=np.linalg.norm(X[i:i+bs,None]-self.R[None],axis=-1); out.append((self.rad[None]/(D+1e-9)).max(1))
        return np.concatenate(out)
MANI_INC=[Manifold(ref_inc,seed=s) for s in range(5)]; MANI_DINO=[Manifold(ref_dino,seed=s) for s in range(5)]
def realism(F_,manis): return np.median(np.stack([m(F_) for m in manis]),0)
def kid(X,Y,n_sub=1000,n_rep=20,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(n_sub,len(X),len(Y)); vals=[]
    for _ in range(n_rep):
        x=X[rng.choice(len(X),m,replace=False)]; y=Y[rng.choice(len(Y),m,replace=False)]
        kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
        vals.append((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
    return float(np.mean(vals)),float(np.std(vals))
def kid_point(X,Y,seed=0):
    rng=np.random.RandomState(seed); dd=X.shape[1]; m=min(len(X),len(Y)); x=X[:m]; y=Y[rng.choice(len(Y),m,replace=False)]
    kk=lambda a,b:(a@b.T/dd+1)**3; kxx,kyy,kxy=kk(x,x),kk(y,y),kk(x,y)
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+(kyy.sum()-np.trace(kyy))/(m*(m-1))-2*kxy.mean())
def boot_ci(x,n=2000,seed=0):
    rng=np.random.RandomState(seed); return np.percentile([np.median(rng.choice(x,len(x))) for _ in range(n)],[2.5,97.5])
MAN_ALL=json.load(open(f'{SCALE}/manifest.json'))['pairs']
KS120=[int(k) for k in pd.read_csv(f'{BASE}/pairs_120/pairs.csv').k]; KSET=set(KS120)
DEV=[int(p_['k']) for p_ in MAN_ALL if int(p_['k']) not in KSET][:24]          # development set: pair-disjoint from the 120 confirmation pairs
print('dev pairs:',DEV); print('confirmation pairs:',len(KS120))

# --- per-pair scoring bundle for a decoded midpoint (human-study proxies) ---
_anc={}
def anchors(k):
    if k in _anc: return _anc[k]
    A,B=pair_imgs(k); rA=to_pil(make_recon(A)); rB=to_pil(make_recon(B)); f=dino_feats([at128(rA),at128(rB)])
    _anc[k]=dict(fA=f[0],fB=f[1],dAB=float(np.linalg.norm(f[0]-f[1])),rA=rA,rB=rB); return _anc[k]
def reencode_field(pil):
    e=cnt_load(pil); ba,bfa,sia=get_gen_inputs(e); sm=build_style_maps(bfa,sia,FMS); L=max(FMS); A=sm[L][0]; Cc=A.shape[0]
    return A.permute(1,2,0).reshape(-1,Cc)
def score_image(k,pil,Z=None,t=None):
    a=anchors(k); im=at128(pil); fi=inc_feats([im])[0]; fd=dino_feats([im])[0]
    r=dict(real_inc=float(realism(fi[None],MANI_INC)[0]),real_dino=float(realism(fd[None],MANI_DINO)[0]),
           dA=float(np.linalg.norm(fd-a['fA'])/a['dAB']),dB=float(np.linalg.norm(fd-a['fB'])/a['dAB']))
    r['place']=r['dA']/(r['dA']+r['dB']+1e-9)
    if Z is not None:
        Xr=reencode_field(pil); r['roundtrip']=float((Xr.double()-Z.double()).norm()/(Z.double().norm()+1e-12))
    return r,fi,fd
_cache_d={}
def get_d(k):
    if k not in _cache_d:
        if len(_cache_d)>64: _cache_d.clear()
        A,B=pair_imgs(k); _cache_d[k]=build_d(cnt_load(A),cnt_load(B))
    return _cache_d[k]
def save_rows(rows,path): pd.DataFrame(rows).to_csv(path,index=False)
def load_rows(path,keys):
    if os.path.exists(path):
        df=pd.read_csv(path); return df.to_dict('records'), set(tuple(r[c] for c in keys) for r in df.to_dict('records'))
    return [],set()
print(f'setup done in {time.time()-t0:.0f}s')

In [ ]:
# ============================================================
# 1. E0 AUDIT (H6) — is the intervention real? tensors consumed per SPADE stage, controls, Sinkhorn convergence, determinism
# ============================================================
k=DEV[0]; d=get_d(k); audit={}
# (a) hooks: record what each SPADE stage actually consumes
consumed=[]
def _hook(i):
    def h(mod,inp,out):
        sm=(inp[1] if len(inp)>1 else out).detach(); consumed.append(dict(stage=i,shape=tuple(sm.shape),norm=float(sm.norm()),var=float(sm.var()),
                                                 sha=hashlib.sha1(np.round(sm.cpu().numpy(),4).tobytes()).hexdigest()[:10]))
    return h
hs=[blk.register_forward_hook(_hook(i)) for i,blk in enumerate(model.generator.spade_blocks)]
imgs={}
for e in [0.005,0.05,0.5]:
    consumed.clear(); fd=field(d,'ot',k,eps=e); Z=zt(fd,0.5); out=decode_field(d,fd,Z); imgs[e]=to_pil(out)
    audit[f'eps{e}']=[dict(c) for c in consumed]
stages=len(audit['eps0.05']); print(f'{stages} SPADE stages consume a style map. Per-stage hashes (must differ across eps):')
for i in range(stages): print(f'  stage {i} shape {audit["eps0.05"][i]["shape"]}: '+'  '.join(f'eps{e}={audit[f"eps{e}"][i]["sha"]}' for e in [0.005,0.05,0.5]))
all_differ=all(len({audit[f'eps{e}'][i]['sha'] for e in [0.005,0.05,0.5]})==3 for i in range(stages))
print('every stage consumes an eps-dependent tensor:',all_differ)
print('image LPIPS  eps.005 vs .05: %.3f | .5 vs .05: %.3f'%(lpips_d(imgs[0.005],imgs[0.05]),lpips_d(imgs[0.5],imgs[0.05])))
# (b) controls: zero / target-mean / large perturbation must change decoder inputs and (usually) the image
fd=field(d,'ot',k); Z=zt(fd,0.5); base=to_pil(decode_field(d,fd,Z)); Y=fd['Y']
ctrl={'zero':torch.zeros_like(Z),'target_mean':Y.mean(0,keepdim=True).expand_as(Z).clone(),'perturb_x3':Z+3*Z.std()*torch.randn_like(Z)}
for n_,Zc in ctrl.items(): print(f'  control {n_:12}: LPIPS vs base = {lpips_d(to_pil(decode_field(d,fd,Zc)),base):.3f}')
# (c) subspace facts: partition of unity, X≈WF, rank
Wm,Fm=splat_W(d,'a'); Q,rank=projector(Wm); X=fd['X'].double()
print(f'W: N={Wm.shape[0]} K={Wm.shape[1]} rank={rank} | row-sum mean {float(Wm.sum(1).mean()):.4f} min {float(Wm.sum(1).min()):.4f} max {float(Wm.sum(1).max()):.4f}')
print(f'X≈WF relative residual: {float((X-Wm@Fm).norm()/X.norm()):.2e} | ||(I-Π)X||/||X|| = {float((X-proj(Q,X)).norm()/X.norm()):.2e}')
# (d) Sinkhorn: canonical 100-iter multiplicative vs converged log-domain — marginal residuals and map difference
L,Cc,H,W,sf,tf=_grids(d); i_s=_sub(len(sf),k,'s'); i_t=_sub(len(tf),k,'t'); src,tgt=sf[i_s],tf[i_t]
for e in [0.005,0.05,0.5]:
    P,_=sinkhorn_plan(src,tgt,e); n=P.shape[0]; res_c=float(max((P.sum(1)-1/n).abs().max()*n,(P.sum(0)-1/n).abs().max()*n))
    Pl,f_,g_,_,res_l,it=sinkhorn_log(src,tgt,e); Tc=(P/(P.sum(1,keepdim=True)+1e-30))@tgt; Tl=((Pl/(Pl.sum(1,keepdim=True)))@tgt.double()).float()
    sc=interaction_scale(src,tgt)
    print(f'  eps={e}: canonical residual {res_c:.2e} | log-domain residual {res_l:.2e} in {it} it | map diff {float((Tc-Tl).norm()/Tl.norm()):.3e} | eps/interaction_rms={e/sc["inter_rms"]:.3f} eps/cost_med={e/sc["cost_med"]:.3f} | H_row={plan_stats(Pl)["H_row"]:.2f} MI={plan_stats(Pl)["MI_nats"]:.3f} nats')
# (e) determinism: same seeds -> identical image
h1=hashlib.sha1(np.asarray(to_pil(decode_field(d,field(d,'ot',k),zt(field(d,'ot',k),0.5)))).tobytes()).hexdigest()
h2=hashlib.sha1(np.asarray(to_pil(decode_field(d,field(d,'ot',k),zt(field(d,'ot',k),0.5)))).tobytes()).hexdigest()
print('deterministic across repeated calls:',h1==h2)
for h_ in hs: h_.remove()
json.dump(dict(pair=k,stages=stages,all_stages_eps_dependent=all_differ,rank=rank,K=Wm.shape[1],N=Wm.shape[0],deterministic=h1==h2,cnt_commit=CNT_COMMIT),open(f'{MECH}/E0_audit.json','w'),indent=1)
print('E0 saved')

In [ ]:
# ============================================================
# 2. E2 SUBSPACE GEOMETRY (H1) — per rule: e_sub, e_aff, neighbour discontinuity, variance retention; does e_sub predict scores?
#    (a) on the 200 pairs of the matching-rule ablation (merged with their saved per-pair scores); (b) commutation error on dev pairs
# ============================================================
RULES2=['meanshift','uniform','gauss','ot_soft','ot','ot_sharp','exact','nn','random']
dfr=pd.read_csv(f'{OUT}/rules_n200.csv'); KS200=sorted(dfr.k.unique())
CSV=f'{MECH}/E2_geometry.csv'; rows,have=load_rows(CSV,['k','rule']); t0=time.time()
for n_,k in enumerate(KS200):
    if all((k,r) in have for r in RULES2): continue
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,rank=projector(Wm)
    for r in RULES2:
        if (k,r) in have: continue
        fd=field(d,r,k); U=fd['U']; g=e_sub(Q,U); a=e_aff(fd['X'],U)
        rows.append(dict(k=k,rule=r,rank=rank,**g,**a,disc=neighbor_disc(U,fd['H'],fd['W'])/neighbor_disc(fd['X'],fd['H'],fd['W']),
                         var_ret=var_ret(U,fd['Y']),e_sub_Y=e_sub(Q,fd['Y'])['e_sub'],**{kk:v for kk,v in fd['info'].items() if isinstance(v,(int,float))}))
    if n_%20==0: save_rows(rows,CSV); print(f'{n_}/{len(KS200)} {time.time()-t0:.0f}s')
save_rows(rows,CSV); g2=pd.DataFrame(rows)
# merge with the ablation's per-pair scores (rule names match; ablation had no meanshift/ot_soft)
sc2=dfr.rename(columns={'rule':'rule'}); m=g2.merge(sc2[['k','rule','real_inc','real_dino','place','endpoint_gap']],on=['k','rule'],how='left')
kidr=json.load(open(f'{OUT}/kid_by_rule.json')) if os.path.exists(f'{OUT}/kid_by_rule.json') else {}
fidr=json.load(open(f'{OUT}/fid_by_rule.json')) if os.path.exists(f'{OUT}/fid_by_rule.json') else {}
print(f"\n{'rule':10}{'e_sub':>8}{'e_aff':>8}{'disc':>7}{'varRet':>8}{'KID':>9}{'FID':>7}{'realDINO':>9}{'|pl-.5|':>8}")
tab={}
for r in RULES2:
    s=m[m.rule==r]; K_=kidr.get(r,[np.nan])[0]; F_=fidr.get(r,[np.nan])[0]
    tab[r]=dict(e_sub=s.e_sub.median(),e_aff=s.e_aff.median(),disc=s.disc.median(),var_ret=s.var_ret.median(),KID=K_,FID=F_,real_dino=s.real_dino.median(),place=(s.place-.5).abs().median())
    print(f"{r:10}{tab[r]['e_sub']:8.3f}{tab[r]['e_aff']:8.3f}{tab[r]['disc']:7.2f}{tab[r]['var_ret']:8.3f}{K_:9.4f}{F_:7.1f}{tab[r]['real_dino']:9.3f}{tab[r]['place']:8.3f}")
print('\ntarget cloud itself outside A-subspace, e_sub(Y) median: %.3f  (what a perfect target copy would cost)'%g2.e_sub_Y.median())
# rule-level: does e_sub rank KID?  pair-level: within-pair Spearman of e_sub vs realism across rules
rr=[r for r in RULES2 if r in kidr]; print('rule-level Spearman(e_sub, KID) = %.2f  (n=%d rules)'%(spearmanr([tab[r]['e_sub'] for r in rr],[tab[r]['KID'] for r in rr])[0],len(rr)))
wp=[spearmanr(s.e_sub,s.real_dino)[0] for k,s in m.dropna(subset=['real_dino']).groupby('k') if len(s)>=5]
print('within-pair Spearman(e_sub, realDINO): median %.2f, frac<0: %.2f  (n=%d pairs)'%(np.median(wp),np.mean(np.array(wp)<0),len(wp)))
# (b) commutation error T(WF)-W T(F) on dev pairs for maps with a natural extension: gauss (affine), meanshift (affine), ot via dual-potential extension
crow=[]
for k in DEV[:12]:
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm); L,Cc,H,W,sf,tf=_grids(d)
    i_s,tt,info=_targets(sf,tf,'gauss',k); A_,(mx,my)=info['A'],info['mu']; Tg=lambda x:(my+(x.double()-mx)@A_.T).float()
    crow.append(dict(k=k,rule='gauss',**commutation_err(Q,Wm,Fm,Tg)))
    mu_s=(tf[_sub(len(tf),k,'t')].mean(0)-sf[_sub(len(sf),k,'s')].mean(0)); crow.append(dict(k=k,rule='meanshift',**commutation_err(Q,Wm,Fm,lambda x:x+mu_s)))
    for e in [0.05,0.5]:
        i_s,tt,info=_targets(sf,tf,'ot',k,eps=e,plan_kind='log'); y=tf[_sub(len(tf),k,'t')]
        Tn=lambda x,g=info['g'],e=e:natural_extension(x,y,g,e).float(); crow.append(dict(k=k,rule=f'ot_natural_eps{e}',**commutation_err(Q,Wm,Fm,Tn)))
cdf=pd.DataFrame(crow); cdf.to_csv(f'{MECH}/E2_commutation.csv',index=False)
print('\ncommutation error T(WF)-W T(F), median over dev pairs (0 = exactly commutes):'); print(cdf.groupby('rule').comm.median().round(4))
json.dump(tab,open(f'{MECH}/E2_table.json','w'),indent=1); print('E2 saved')

In [ ]:
# ============================================================
# 3. E1 TIME × SMOOTHING GRID (H2) — is ε-robustness specific to t≈0.5? geometry + decoded scores on 24 dev pairs
# ============================================================
CONDS=[('ot',dict(eps=e)) for e in [0.005,0.01,0.05,0.1,0.5]]+[('uniform',{}),('exact',{})]
TS=[0,0.25,0.5,0.75,0.9,1.0]; REF=('ot',0.05)
CSV=f'{MECH}/E1_grid.csv'; rows,have=load_rows(CSV,['k','cond','t']); t0=time.time()
def cname(r,kw): return f"{r}{kw.get('eps','')}"
for n_,k in enumerate(DEV):
    if all((k,cname(r,kw),t) in have for r,kw in CONDS for t in TS): continue
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm); ref_imgs={}
    fds={cname(r,kw):field(d,r,k,**kw) for r,kw in CONDS}
    for t in TS: ref_imgs[t]=to_pil(decode_field(d,fds['ot0.05'],zt(fds['ot0.05'],t)))
    for r,kw in CONDS:
        c=cname(r,kw); fd=fds[c]; geo=dict(**e_sub(Q,fd['U']),var_ret=var_ret(fd['U'],fd['Y']),**{kk:v for kk,v in fd['info'].items() if isinstance(v,(int,float))})
        if r=='ot': geo.update(interaction_scale(fd['X'][fd['i_s']],fd['Y'][_sub(len(fd['Y']),k,'t')]))
        for t in TS:
            if (k,c,t) in have: continue
            Z=zt(fd,t); pil=to_pil(decode_field(d,fd,Z)); s,_,_=score_image(k,pil,Z=Z)
            rows.append(dict(k=k,cond=c,t=t,**geo,**s,e_sub_Z=e_sub(Q,Z)['e_sub'],lpips_ref=lpips_d(pil,ref_imgs[t])))
    save_rows(rows,CSV); print(f'{n_+1}/{len(DEV)} {time.time()-t0:.0f}s')
g1=pd.DataFrame(rows); g1.to_csv(CSV,index=False)
piv=lambda col: g1.pivot_table(index='cond',columns='t',values=col,aggfunc='median').round(3)
print('\nLPIPS to the ε=0.05 path at the same t (0 at ε=0.05 by definition):'); print(piv('lpips_ref'))
print('\nDINO realism:'); print(piv('real_dino'))
print('\nnormalized distance to reconstructed B (dB; t=1 column = endpoint gap):'); print(piv('dB'))
print('\nround-trip discrepancy (re-encode decoded image vs intended Z_t):'); print(piv('roundtrip'))
print('\ne_sub of the target field U per condition (t-independent):'); print(g1.groupby('cond').e_sub.median().round(3))
# interaction test: does the ε effect (LPIPS to reference) grow with t?  (H2 predicts yes)
sub=g1[g1.cond!='ot0.05']; print('\nmedian LPIPS-to-ref by t (all non-reference conditions):'); print(sub.groupby('t').lpips_ref.median().round(3))
print('E1 saved')

In [ ]:
# ============================================================
# 4. E3 CAUSAL PROJECTION (H1) — remove/add the off-subspace component of the hard map; matched controls; 24 dev pairs
# ============================================================
from sklearn.cluster import KMeans
_km={}
def motif_cov(k,Y,Zre,kc=8):
    """1 - TV distance between target-cluster histograms of Y and of a feature cloud Zre (both assigned to Y's k-means centroids)."""
    if k not in _km: _km[k]=KMeans(kc,n_init=4,random_state=0).fit(Y.detach().cpu().numpy())
    km=_km[k]; hY=np.bincount(km.labels_,minlength=kc)/len(km.labels_); lab=km.predict(Zre.detach().cpu().numpy().astype(np.float32)); hZ=np.bincount(lab,minlength=kc)/len(lab)
    return float(1-0.5*np.abs(hY-hZ).sum())
def blur_field(U,H,W,sigma):
    G=U.reshape(H,W,-1).permute(2,0,1)[None].float(); r=int(3*sigma)+1; x=torch.arange(-r,r+1,device=U.device).float(); w=torch.exp(-x**2/(2*sigma**2)); w=w/w.sum()
    G=F.conv2d(F.pad(G,(r,r,0,0),mode='reflect'),w.view(1,1,1,-1).repeat(G.shape[1],1,1,1),groups=G.shape[1])
    G=F.conv2d(F.pad(G,(0,0,r,r),mode='reflect'),w.view(1,1,-1,1).repeat(G.shape[1],1,1,1),groups=G.shape[1])
    return G[0].permute(1,2,0).reshape(H*W,-1)
LAMS=[0,0.25,0.5,1.0]; TS3=[0.5,0.9,1.0]
CSV=f'{MECH}/E3_projection.csv'; rows,have=load_rows(CSV,['k','variant','t']); t0=time.time()
def run(k,d,fd,name,Z_of_t,Q):
    for t in TS3:
        if (k,name,t) in have: continue
        Z=Z_of_t(t); pil=to_pil(decode_field(d,fd,Z)); s,_,_=score_image(k,pil,Z=Z); r=dict(k=k,variant=name,t=t,**s,e_sub_Z=e_sub(Q,Z)['e_sub'])
        if t==1.0: r['motif_cov']=motif_cov(k,fd['Y'],reencode_field(pil))
        rows.append(r)
for n_,k in enumerate(DEV):
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm)
    fh=field(d,'exact',k); X=fh['X'].double(); U=fh['U'].double(); Up=proj(Q,U); Uo=U-Up; nrm=float(Uo.norm())
    fo=field(d,'ot',k); Uot=fo['U'].double()
    g=torch.Generator(device=device).manual_seed(k+101); rnd=torch.randn_like(U,generator=g)
    Uin=proj(Q,rnd); Uin=Uin/Uin.norm()*nrm                                   # equal-norm in-subspace perturbation
    Uoff=(rnd-proj(Q,rnd)); Uoff=Uoff/Uoff.norm()*nrm                         # equal-norm off-subspace random (control for 'specific' vs 'any' off component)
    # low-pass control matched in perturbation magnitude
    sig=0.5
    for _ in range(30):
        Ub=blur_field(U.float(),fh['H'],fh['W'],sig).double(); dn=float((U-Ub).norm())
        if abs(dn-nrm)/nrm<0.05: break
        sig=min(sig*1.15,8.0) if dn<nrm else sig*0.87
    # variants (all decoded with A's layout)
    V={}
    for lam in LAMS: V[f'hard_lam{lam}']=lambda t,lam=lam:(1-t)*X+t*(Up+lam*Uo)
    V['hard_inSubCtrl']=lambda t:(1-t)*X+t*(Up+Uin)                          # projection + equal-norm in-subspace noise
    V['hard_offRandCtrl']=lambda t:(1-t)*X+t*(Up+Uoff)                       # projection + equal-norm random off-subspace noise
    V['hard_blurCtrl']=lambda t,Ub=Ub:(1-t)*X+t*Ub                          # generic smoothing, matched magnitude
    V['ot_base']=lambda t:(1-t)*X+t*Uot
    for a_ in [0.5,1.0]: V[f'ot_plusPerp{a_}']=lambda t,a_=a_:(1-t)*X+t*(Uot+a_*Uo)   # add-back dose-response
    V['ot_plusInSub1.0']=lambda t:(1-t)*X+t*(Uot+Uin)
    for rname in ['random','pixel_pos']:
        fr=field(d,'random',k) if rname=='random' else dict(fh,U=fh['Y'])   # positional: U = target grid itself
        Ur=fr['U'].double(); Urp=proj(Q,Ur)
        V[f'{rname}_lam1']=lambda t,Ur=Ur:(1-t)*X+t*Ur; V[f'{rname}_lam0']=lambda t,Urp=Urp:(1-t)*X+t*Urp
    for name,Zf in V.items(): run(k,d,fh,name,Zf,Q)
    save_rows(rows,CSV); print(f'{n_+1}/{len(DEV)} {time.time()-t0:.0f}s')
g3=pd.DataFrame(rows); g3.to_csv(CSV,index=False)
for col in ['real_dino','real_inc','roundtrip','place','dB','e_sub_Z']:
    print(f'\n{col} (median over dev pairs):'); print(g3.pivot_table(index='variant',columns='t',values=col,aggfunc='median').round(3))
print('\nmotif coverage at t=1 (1 = target cluster histogram reproduced after decode+re-encode):'); print(g3[g3.t==1].groupby('variant').motif_cov.median().round(3))
# paired tests: projection vs hard, and vs equal-norm controls, at t=0.5
p=g3[g3.t==0.5].pivot(index='k',columns='variant',values='real_dino')
for a_,b_ in [('hard_lam0','hard_lam1.0'),('hard_lam0.25','hard_lam1.0'),('hard_lam0','hard_inSubCtrl'),('hard_lam0','hard_blurCtrl'),('hard_lam0','hard_offRandCtrl'),('ot_plusPerp1.0','ot_base'),('ot_plusInSub1.0','ot_base')]:
    dlt=(p[a_]-p[b_]).dropna().values; lo,hi=boot_ci(dlt); print(f'  realDINO {a_:>16} - {b_:<16}: {np.median(dlt):+.3f} [{lo:+.3f},{hi:+.3f}] p={wilcoxon(dlt).pvalue:.2g}')
# freeze lambda* for E7: best median realDINO gain over hard at t=0.5 with placement not worse than hard by >0.01
pp=g3[g3.t==0.5].pivot(index='k',columns='variant',values='place'); best=None
for lam in [0,0.25,0.5]:
    v=f'hard_lam{lam}'; gain=float(np.median((p[v]-p['hard_lam1.0']).dropna())); dpl=float(np.median(((pp[v]-.5).abs()-(pp['hard_lam1.0']-.5).abs()).dropna()))
    print(f'  lam={lam}: realDINO gain {gain:+.3f}, |place-.5| change {dpl:+.3f}')
    if dpl<=0.01 and (best is None or gain>best[1]): best=(lam,gain)
LAM_STAR=best[0] if best else 0.25; json.dump(dict(lam_star=LAM_STAR,rule='max median realDINO gain over hard at t=.5 s.t. |place-.5| not worse by >0.01'),open(f'{MECH}/lambda_star.json','w'))
print('frozen lambda* =',LAM_STAR,'| E3 saved')

In [ ]:
# ============================================================
# 5. E4 AMPLITUDE / EXTENSION / BUDGET (H4) — is the hard–soft gap smoothness, residual amplitude, or the NN extension?
# ============================================================
TS4=[0.5,1.0]
CSV=f'{MECH}/E4_extension.csv'; rows,have=load_rows(CSV,['k','variant','t']); t0=time.time()
def add(k,d,fd,name,U,Q,t):
    if (k,name,t) in have: return
    Z=(1-t)*fd['X'].double()+t*U.double(); pil=to_pil(decode_field(d,fd,Z)); s,_,_=score_image(k,pil,Z=Z)
    rows.append(dict(k=k,variant=name,t=t,**s,e_sub=e_sub(Q,U)['e_sub'],disc=neighbor_disc(U.float(),fd['H'],fd['W'])/neighbor_disc(fd['X'],fd['H'],fd['W']),var_ret=var_ret(U,fd['Y'])))
for n_,k in enumerate(DEV):
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm); L,Cc,H,W,sf,tf=_grids(d)
    # (i) residual amplitude: T = mu + R ; original / R rescaled to the hard map's RMS / R rows shuffled (same amplitude, source-conditioning destroyed)
    fh=field(d,'exact',k); Rh=fh['U'].double()-fh['U'].double().mean(0); rms_h=float(Rh.pow(2).mean().sqrt())
    for name,fd in [('ot0.005',field(d,'ot',k,eps=0.005)),('ot0.05',field(d,'ot',k)),('ot0.5',field(d,'ot',k,eps=0.5)),('hard',fh)]:
        U=fd['U'].double(); mu=U.mean(0,keepdim=True); R=U-mu; rms=float(R.pow(2).mean().sqrt())
        g=torch.Generator(device=device).manual_seed(k+5); perm=torch.randperm(len(R),generator=g,device=device)
        for t in TS4:
            add(k,d,fd,f'{name}_orig',U,Q,t); add(k,d,fd,f'{name}_rmsMatched',mu+R*(rms_h/rms),Q,t); add(k,d,fd,f'{name}_shuffled',mu+R[perm],Q,t)
    # (ii) extension: canonical nn extension vs natural dual-potential extension of the SAME converged soft plan
    for e in [0.05,0.5]:
        i_s,tt,info=_targets(sf,tf,'ot',k,eps=e,plan_kind='log'); y=tf[_sub(len(tf),k,'t')]
        nn_=torch.cdist(sf,sf[i_s]).argmin(1); U_nn=tt[nn_]; U_nat=natural_extension(sf,y,info['g'],e).float()
        fd=dict(X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,U=U_nn)
        for t in TS4: add(k,d,fd,f'ot{e}_nnExt',U_nn,Q,t); add(k,d,fd,f'ot{e}_naturalExt',U_nat,Q,t)
        if (k,f'ot{e}_extLPIPS',0.5) not in have:
            pil_nn=to_pil(decode_field(d,fd,(1-.5)*sf+.5*U_nn)); pil_nat=to_pil(decode_field(d,fd,(1-.5)*sf+.5*U_nat)); rows.append(dict(k=k,variant=f'ot{e}_extLPIPS',t=0.5,lpips=lpips_d(pil_nn,pil_nat)))
    # (iii) budget: hard and soft at matched sample budgets
    for M_ in [1024,4096]:
        fd=field(d,'ot',k,M=M_); [add(k,d,fd,f'ot0.05_M{M_}',fd['U'],Q,t) for t in TS4]
    for ne in [1024,2048]:
        fd=field(d,'exact',k,n_exact=ne); [add(k,d,fd,f'hard_n{ne}',fd['U'],Q,t) for t in TS4]
    save_rows(rows,CSV); print(f'{n_+1}/{len(DEV)} {time.time()-t0:.0f}s')
g4=pd.DataFrame(rows); g4.to_csv(CSV,index=False)
for col in ['real_dino','real_inc','roundtrip','dB','e_sub','disc','var_ret']:
    print(f'\n{col}:'); print(g4[g4.variant.str.contains('LPIPS')==False].pivot_table(index='variant',columns='t',values=col,aggfunc='median').round(3))
print('\nLPIPS between nn-extension and natural-extension decodes (t=0.5):'); print(g4[g4.variant.str.contains('extLPIPS')].groupby('variant').lpips.median().round(3))
print('E4 saved')

In [ ]:
# ============================================================
# 6. E6 TARGET INFORMATION LOST BY SMOOTHING — law of total covariance, conditional cluster entropy,
#    rare-cluster coverage, motif recovery after decode+re-encode. Resumable per (k, cond).
# ============================================================
from sklearn.cluster import KMeans
_km = globals().get('_km', {})
CSV = f'{MECH}/E6_information.csv'; rows, have = load_rows(CSV, ['k','cond']); t0 = time.time()
KC = 8                                     # target clusters (motifs)

for n_, k in enumerate(DEV):
    k = int(k)
    d = get_d(k); L, Cc, H, W, sf, tf = _grids(d)
    i_s = _sub(len(sf), k, 's'); i_t = _sub(len(tf), k, 't'); src, tgt = sf[i_s], tf[i_t]
    Y = tgt.double(); SigB = float(Y.var(0, unbiased=False).sum())

    # cluster the FULL target cloud once per pair; reuse the same KMeans as E3 if it is cached
    if k not in _km:
        _km[k] = KMeans(KC, n_init=4, random_state=0).fit(tf.detach().cpu().numpy().astype(np.float32))
    km = _km[k]
    lab_t = torch.as_tensor(km.predict(tgt.detach().cpu().numpy().astype(np.float32)),
                            device=device).long()
    onehot = F.one_hot(lab_t, KC).double(); hY = onehot.mean(0)
    hY_np = hY.cpu().numpy(); rare_c = int(hY.argmin()); rareY = float(hY.min())
    H_marg = float(-(hY * torch.log(hY.clamp_min(1e-300))).sum())

    def hist_tv(cloud):
        """total-variation distance between a cloud's target-cluster histogram and the target's own."""
        lab = km.predict(cloud.detach().cpu().numpy().astype(np.float32))
        h = np.bincount(lab, minlength=KC) / len(lab)
        return float(0.5 * np.abs(h - hY_np).sum())

    # decoder-loss baseline: what the cluster histogram looks like after B is merely reconstructed
    if (k, 'reconB') not in have:
        rows.append(dict(k=k, cond='reconB', tv_reencoded=hist_tv(reencode_field(anchors(k)['rB']))))

    # ---- entropic maps across eps: exact conditional-covariance decomposition + motif measures ----
    for e in [0.005, 0.01, 0.05, 0.1, 0.5]:
        c = f'ot{e}'
        if (k, c) in have: continue
        P, f_, g_, _, res, it = sinkhorn_log(src, tgt, e)
        q = P / P.sum(1, keepdim=True)                       # row-conditional distribution
        T = q @ Y                                            # barycentric projection
        covT = float(T.var(0, unbiased=False).sum())
        EcondCov = float(((q @ (Y * Y).sum(1, keepdim=True)) - (T * T).sum(1, keepdim=True)).mean())
        qc = q @ onehot                                      # conditional cluster distribution per source point
        Hc = float(-(qc * torch.log(qc.clamp_min(1e-300))).sum(1).mean())
        rare_proj = float(qc[:, rare_c].mean())
        # decode the eta=1 field built from this map, then re-encode and measure motif recovery
        fd = dict(X=sf, Y=tf, L=L, Cc=Cc, H=H, W=W, U=T.float()[torch.cdist(sf, src).argmin(1)])
        pil = to_pil(decode_field(d, fd, fd['U']))
        rows.append(dict(k=k, cond=c, eps=e,
                         covT_frac=covT / SigB, condCov_frac=EcondCov / SigB,
                         identity_err=abs(covT + EcondCov - SigB) / SigB,
                         H_cluster_cond=Hc, H_cluster_marg=H_marg,
                         rare_share_Y=rareY, rare_share_proj=rare_proj,
                         tv_projected=hist_tv(T), tv_reencoded=hist_tv(reencode_field(pil)),
                         sinkhorn_res=res, sinkhorn_iters=it, **plan_stats(P)))

    # ---- non-entropic rules, same measures where they are defined ----
    for c in ['exact', 'gauss', 'meanshift', 'uniform']:
        if (k, c) in have: continue
        fd = field(d, c, k)
        pil = to_pil(decode_field(d, fd, fd['U']))
        rows.append(dict(k=k, cond=c,
                         tv_projected=hist_tv(fd['tt']), tv_reencoded=hist_tv(reencode_field(pil)),
                         covT_frac=var_ret(fd['tt'], tgt)))

    save_rows(rows, CSV); print(f'{n_+1}/{len(DEV)}  {time.time()-t0:.0f}s')

g6 = pd.DataFrame(rows); g6.to_csv(CSV, index=False)
print('\nlaw of total covariance, max relative error over pairs/eps:', float(g6.identity_err.max()))
cols = ['covT_frac','condCov_frac','H_cluster_cond','rare_share_proj','tv_projected','tv_reencoded','MI_nats']
print('\nmedians by condition:')
print('  covT_frac      = share of target variance kept by the barycentric projection')
print('  condCov_frac   = share discarded (E[Cov(Y|X)]); the two must sum to 1')
print('  H_cluster_cond = conditional motif entropy (marginal = %.2f nats)' % g6.H_cluster_marg.median())
print('  tv_projected   = motif-histogram distance in feature space; tv_reencoded = after decode+re-encode (0 = motifs reproduced)')
print(g6.groupby('cond')[[c for c in cols if c in g6]].median().round(3))
print('\nrare target cluster share in Y (median): %.3f' % g6.rare_share_Y.median())
print('decoder-loss baseline (reconB) tv_reencoded: %.3f  <- charge anything below this to the decoder, not to OT'
      % g6[g6.cond=='reconB'].tv_reencoded.median())
print('E6 saved')

In [ ]:
# ============================================================
# 7. E5 DECODER SENSITIVITY (H3) — finite-difference response to matched-norm perturbation directions; per-stage sharp/smooth swaps
# ============================================================
CSV=f'{MECH}/E5_sensitivity.csv'; rows,have=load_rows(CSV,['k','direction','h']); t0=time.time()
HS=[0.02,0.05,0.1]
for n_,k in enumerate(DEV[:12]):
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm); fo=field(d,'ot',k); fh=field(d,'exact',k); X=fo['X'].double()
    Z=(0.5*X+0.5*fo['U'].double()); base=to_pil(decode_field(d,fo,Z)); fb=dino_feats([at128(base)])[0]; nZ=float(Z.norm())
    g=torch.Generator(device=device).manual_seed(k+9); rnd=torch.randn_like(Z,generator=g)
    dirs={'sharp_minus_smooth':fh['U'].double()-fo['U'].double(),'hard_offSub':fh['U'].double()-proj(Q,fh['U'].double()),
          'inSub_random':proj(Q,rnd),'offSub_random':rnd-proj(Q,rnd),'centered_amplitude':Z-Z.mean(0),'random':rnd}
    for name,v in dirs.items():
        v=v/v.norm()
        for h in HS:
            if (k,name,h) in have: continue
            pil=to_pil(decode_field(d,fo,Z+h*nZ*v)); fd_=dino_feats([at128(pil)])[0]
            pix=float(np.sqrt(((np.asarray(pil,dtype=np.float32)-np.asarray(base,dtype=np.float32))**2).mean()))/255.
            rows.append(dict(k=k,direction=name,h=h,resp_pix=pix/h,resp_lpips=lpips_d(pil,base)/h,resp_dino=float(np.linalg.norm(fd_-fb))/h))
    # per-stage swap: smooth conditioning at all stages except sharp at stage i (and the reverse)
    Zs=0.5*X+0.5*fh['U'].double(); sm_s=sm_from_field(Z.float(),fo['L'],fo['H'],fo['W'],fo['Cc']); sm_h=sm_from_field(Zs.float(),fo['L'],fo['H'],fo['W'],fo['Cc'])
    nst=len(model.generator.spade_blocks); all_s=to_pil(decode_stagewise(d['blob_a'],[sm_s]*nst)); all_h=to_pil(decode_stagewise(d['blob_a'],[sm_h]*nst)); fs_=dino_feats([at128(all_s),at128(all_h)])
    for i in range(nst):
        if (k,f'stage{i}_sharp',0) in have: continue
        mix=[sm_h if j==i else sm_s for j in range(nst)]; pil=to_pil(decode_stagewise(d['blob_a'],mix)); fm_=dino_feats([at128(pil)])[0]
        rows.append(dict(k=k,direction=f'stage{i}_sharp',h=0,lpips_vs_smooth=lpips_d(pil,all_s),lpips_vs_sharp=lpips_d(pil,all_h),dino_vs_smooth=float(np.linalg.norm(fm_-fs_[0])),dino_vs_sharp=float(np.linalg.norm(fm_-fs_[1]))))
    if (k,'allSharp_vs_allSmooth',0) not in have: rows.append(dict(k=k,direction='allSharp_vs_allSmooth',h=0,lpips_vs_smooth=lpips_d(all_h,all_s),dino_vs_smooth=float(np.linalg.norm(fs_[1]-fs_[0]))))
    save_rows(rows,CSV); print(f'{n_+1}/12 {time.time()-t0:.0f}s')
g5=pd.DataFrame(rows); g5.to_csv(CSV,index=False)
print('\nresponse per unit relative perturbation (median over pairs); H3 predicts off-subspace >> in-subspace at equal norm:')
print(g5[g5.h>0].pivot_table(index='direction',columns='h',values=['resp_lpips','resp_dino'],aggfunc='median').round(3))
print('\nper-stage swap (sharp at one stage only): distance to all-smooth decode; which stage carries the sharp/smooth difference?')
print(g5[g5.h==0].groupby('direction')[['lpips_vs_smooth','dino_vs_smooth']].median().round(3))
print('E5 saved')

In [ ]:
# ============================================================
# 8. E7 CONFIRMATION — frozen λ* projected transport vs baselines on the frozen 120 pairs; KID with paired pair-resampling (shared draws)
# ============================================================
LAM_STAR=json.load(open(f'{MECH}/lambda_star.json'))['lam_star'] if os.path.exists(f'{MECH}/lambda_star.json') else 0.25
print('lambda* =',LAM_STAR)
NEW={'proj_hard':f'{SCALE}/proj_hard_lam{LAM_STAR}','meanshift':f'{SCALE}/meanshift','target_mean':f'{SCALE}/target_mean','source_copy':f'{SCALE}/source_copy','target_copy':f'{SCALE}/target_copy','unrelated_real':f'{SCALE}/unrelated_real'}
for p in NEW.values(): os.makedirs(p,exist_ok=True)
rng=np.random.RandomState(2026); all_dtd=sorted(glob.glob(f'{LOCAL}/*/*.jpg')); t0=time.time()
for n_,k in enumerate(KS120):
    if all(os.path.exists(f'{p}/{k:04d}.png') for p in NEW.values()): continue
    d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm); fh=field(d,'exact',k); X=fh['X'].double(); U=fh['U'].double(); Up=proj(Q,U)
    to_pil(decode_field(d,fh,0.5*X+0.5*(Up+LAM_STAR*(U-Up)))).save(f'{NEW["proj_hard"]}/{k:04d}.png')
    fm=field(d,'meanshift',k); to_pil(decode_field(d,fm,zt(fm,0.5))).save(f'{NEW["meanshift"]}/{k:04d}.png')
    fu=field(d,'uniform',k); to_pil(decode_field(d,fu,zt(fu,0.5))).save(f'{NEW["target_mean"]}/{k:04d}.png')
    a=anchors(k); a['rA'].save(f'{NEW["source_copy"]}/{k:04d}.png'); a['rB'].save(f'{NEW["target_copy"]}/{k:04d}.png')
    Image.open(all_dtd[rng.randint(len(all_dtd))]).convert('RGB').resize((256,256)).save(f'{NEW["unrelated_real"]}/{k:04d}.png')
    if n_%20==0: print(f'gen {n_}/120 {time.time()-t0:.0f}s')
METH={**{m:f'{SCALE}/{m}' for m in ['ot','exact','ot_sym','gaussian_w2','pixel_linear','texton_linear']},'tm':f'{SCALE}/mid',**NEW}
METH={m:p for m,p in METH.items() if all(os.path.exists(f'{p}/{k:04d}.png') for k in KS120)}
anc=dict(np.load(f'{OUT}/anchor_dino.npz')); XI={}; XD={}; rows=[]
for m,pdir in METH.items():
    fI,fD=f'{OUT}/feat_{m}_inc.npy',f'{OUT}/feat_{m}_dino.npy'; ks_all=[int(p_['k']) for p_ in MAN_ALL if os.path.exists(f'{pdir}/{int(p_["k"]):04d}.png')]
    if os.path.exists(fI) and len(ks_all)>=1000 and len(np.load(fI,mmap_mode='r'))==len(ks_all):
        idx=[i for i,k in enumerate(ks_all) if k in KSET]; XI[m]=np.load(fI)[idx]; XD[m]=np.load(fD)[idx]; ks=[ks_all[i] for i in idx]
    else:
        ims=[at128(Image.open(f'{pdir}/{k:04d}.png')) for k in KS120]; XI[m]=inc_feats(ims); XD[m]=dino_feats(ims); ks=KS120[:]
    order=np.argsort(ks); XI[m]=XI[m][order]; XD[m]=XD[m][order]; ks=sorted(ks)
    rI=realism(XI[m],MANI_INC); rD=realism(XD[m],MANI_DINO)
    for i,k in enumerate(ks):
        dA=np.linalg.norm(XD[m][i]-anc[f'{k}_A']); dB=np.linalg.norm(XD[m][i]-anc[f'{k}_B']); rows.append(dict(k=k,method=m,real_inc=rI[i],real_dino=rD[i],place=dA/(dA+dB+1e-9)))
sc7=pd.DataFrame(rows); sc7.to_csv(f'{MECH}/E7_scores.csv',index=False); piv={c:sc7.pivot(index='k',columns='method',values=c) for c in ['real_inc','real_dino','place']}
# KID with paired pair-resampling: the SAME bootstrap pair indices for every method, fixed reference subsample
B=200; rngb=np.random.RandomState(0); refsub=ref_inc[np.random.RandomState(1).choice(len(ref_inc),120,replace=False)]
draws={m:[] for m in METH}; pt={m:kid_point(XI[m],refsub) for m in METH}
for b in range(B):
    idx=rngb.choice(120,120,replace=True)
    for m in METH: draws[m].append(kid_point(XI[m][idx],refsub))
draws={m:np.array(v) for m,v in draws.items()}
print(f"\n=== E7 CONFIRMATION, 120 frozen pairs, canonical 128px; KID CI from paired pair-resampling (shared draws) ===")
print(f"{'method':16}{'KID':>8}{'95% CI':>18}{'dKID vs ot [CI]':>24}{'realInc':>9}{'realDINO':>10}{'|pl-.5|':>9}{'dRealDINO vs ot':>18}")
res7={}
for m in METH:
    lo,hi=np.percentile(draws[m],[2.5,97.5]); dd=draws[m]-draws['ot']; dlo,dhi=np.percentile(dd,[2.5,97.5])
    dr=(piv['real_dino'][m]-piv['real_dino']['ot']).dropna().values; rlo,rhi=boot_ci(dr) if m!='ot' else (0,0)
    res7[m]=dict(KID=pt[m],KID_ci=[lo,hi],dKID_vs_ot=[float(np.mean(dd)),dlo,dhi],RInc=float(piv['real_inc'][m].median()),RDin=float(piv['real_dino'][m].median()),Place=float((piv['place'][m]-.5).abs().median()))
    print(f"{m:16}{pt[m]:8.4f}   [{lo:.4f},{hi:.4f}]   {np.mean(dd):+.4f} [{dlo:+.4f},{dhi:+.4f}]{res7[m]['RInc']:9.3f}{res7[m]['RDin']:10.3f}{res7[m]['Place']:9.3f}   {np.median(dr):+.3f} [{rlo:+.3f},{rhi:+.3f}]")
json.dump(res7,open(f'{MECH}/E7_numbers.json','w'),indent=1); print('E7 saved')

In [ ]:
# ============================================================
# 9. SUMMARY — key tables + figures into mechanism/
# ============================================================
S={}
if os.path.exists(f'{MECH}/E0_audit.json'): S['E0']=json.load(open(f'{MECH}/E0_audit.json'))
if os.path.exists(f'{MECH}/E2_table.json'):
    S['E2']=json.load(open(f'{MECH}/E2_table.json')); tb=pd.DataFrame(S['E2']).T
    fig,ax=plt.subplots(1,2,figsize=(9,3.5))
    for a_,col,lab in [(ax[0],'KID','KID (n=200)'),(ax[1],'real_dino','DINO realism')]:
        a_.scatter(tb.e_sub,tb[col]); [a_.annotate(r,(tb.e_sub[r],tb[col][r]),fontsize=8) for r in tb.index]; a_.set_xlabel('e_sub (off-subspace energy of U)'); a_.set_ylabel(lab)
    plt.tight_layout(); plt.savefig(f'{FIGS}/E2_esub_vs_scores.png',dpi=160); plt.show()
if os.path.exists(f'{MECH}/E1_grid.csv'):
    g1=pd.read_csv(f'{MECH}/E1_grid.csv'); fig,ax=plt.subplots(1,3,figsize=(13,3.5))
    for a_,col in zip(ax,['lpips_ref','dB','roundtrip']):
        pv=g1.pivot_table(index='cond',columns='t',values=col,aggfunc='median'); im=a_.imshow(pv.values,aspect='auto',cmap='viridis'); a_.set_xticks(range(len(pv.columns))); a_.set_xticklabels(pv.columns); a_.set_yticks(range(len(pv.index))); a_.set_yticklabels(pv.index); a_.set_title(col); plt.colorbar(im,ax=a_)
    plt.tight_layout(); plt.savefig(f'{FIGS}/E1_eps_by_t.png',dpi=160); plt.show(); S['E1']={c:{str(t):float(v) for t,v in r.items()} for c,r in g1.pivot_table(index='cond',columns='t',values='lpips_ref',aggfunc='median').iterrows()}
if os.path.exists(f'{MECH}/E3_projection.csv'):
    g3=pd.read_csv(f'{MECH}/E3_projection.csv'); fig,ax=plt.subplots(1,3,figsize=(13,3.5))
    for a_,col in zip(ax,['real_dino','place','dB']):
        for t in [0.5,0.9,1.0]:
            s=g3[(g3.t==t)&g3.variant.str.startswith('hard_lam')].groupby('variant')[col].median(); lam=[float(v.replace('hard_lam','')) for v in s.index]; a_.plot(lam,s.values,'o-',label=f't={t}')
        a_.set_xlabel('λ (off-subspace fraction kept)'); a_.set_title(col); a_.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(f'{FIGS}/E3_lambda_sweep.png',dpi=160); plt.show()
    S['E3']=g3.pivot_table(index='variant',columns='t',values='real_dino',aggfunc='median').round(3).to_dict()
for name in ['E4_extension','E6_information','E5_sensitivity']:
    if os.path.exists(f'{MECH}/{name}.csv'): S[name]='see csv'
if os.path.exists(f'{MECH}/E7_numbers.json'): S['E7']=json.load(open(f'{MECH}/E7_numbers.json'))
json.dump(S,open(f'{MECH}/summary.json','w'),indent=1,default=float)
print('summary written to',f'{MECH}/summary.json'); print('figures in',FIGS)
print('\n>>> paste back: E0 printout, E2 table + correlations, E1 pivots, E3 paired tests + lambda*, E4/E6/E5 pivots, E7 table <<<')

In [ ]:
# ============================================================
# E2b. FIX — normalized e_sub/e_aff are undefined for near-constant maps; report absolute RMS; Spearman excludes uniform
# ============================================================
g2=pd.read_csv(f'{MECH}/E2_geometry.csv')
near_const=(g2.cen_rms<1e-3*g2.cen_rms.median())|(g2.rule=='uniform')
g2.loc[near_const,['e_sub','e_aff']]=np.nan
g2.to_csv(f'{MECH}/E2_geometry_fixed.csv',index=False)
kidr=json.load(open(f'{OUT}/kid_by_rule.json')) if os.path.exists(f'{OUT}/kid_by_rule.json') else {}
dfr=pd.read_csv(f'{OUT}/rules_n200.csv'); m=g2.merge(dfr[['k','rule','real_inc','real_dino','place']],on=['k','rule'],how='left')
print(f"{'rule':10}{'e_sub':>8}{'off_rms':>9}{'e_aff':>8}{'aff_rms':>9}{'disc':>7}{'varRet':>8}{'KID':>9}{'realDINO':>9}{'|pl-.5|':>8}")
tab={}
for r in ['meanshift','uniform','gauss','ot_soft','ot','ot_sharp','exact','nn','random']:
    s=m[m.rule==r]; K_=kidr.get(r,[np.nan])[0]
    tab[r]=dict(e_sub=s.e_sub.median(),off_rms=s.off_rms.median(),e_aff=s.e_aff.median(),aff_rms=s.aff_rms.median(),disc=s.disc.median(),var_ret=s.var_ret.median(),KID=K_,real_dino=s.real_dino.median(),place=(s.place-.5).abs().median())
    t=tab[r]; f=lambda v,w=8,p=3: (f"{v:{w}.{p}f}" if pd.notna(v) else f"{'undef':>{w}}")
    print(f"{r:10}{f(t['e_sub'])}{f(t['off_rms'],9,4)}{f(t['e_aff'])}{f(t['aff_rms'],9,4)}{f(t['disc'],7,2)}{f(t['var_ret'])}{f(t['KID'],9,4)}{f(t['real_dino'],9)}{f(t['place'])}")
rr=[r for r in tab if r in kidr and pd.notna(tab[r]['e_sub'])]
print('\nrule-level Spearman(e_sub,KID) excluding undefined: %.2f (n=%d)'%(spearmanr([tab[r]['e_sub'] for r in rr],[tab[r]['KID'] for r in rr])[0],len(rr)))
rr2=[r for r in rr if r!='random']
print('same, also excluding random: %.2f (n=%d)'%(spearmanr([tab[r]['e_sub'] for r in rr2],[tab[r]['KID'] for r in rr2])[0],len(rr2)))
json.dump(tab,open(f'{MECH}/E2_table_fixed.json','w'),indent=1,default=float); print('E2b saved')

In [ ]:
# ============================================================
# HELPERS-2 — energy distance, true full-grid affine maps (no NN extension), distributional round-trip
# ============================================================
def energy_dist(X,Y,m=2048,seed=0):
    g=torch.Generator(device=device).manual_seed(seed)
    x=X[torch.randperm(len(X),generator=g,device=device)[:m]].double(); y=Y[torch.randperm(len(Y),generator=g,device=device)[:m]].double()
    return float(2*torch.cdist(x,y).mean()-torch.cdist(x,x).mean()-torch.cdist(y,y).mean())
def affine_full(d,rule,k):
    """Gaussian-W2 or mean-shift fitted on the canonical subsample, APPLIED TO ALL N CELLS (exactly affine, commutes with splatting)."""
    L,Cc,H,W,sf,tf=_grids(d)
    if rule=='gauss_full':
        _,_,info=_targets(sf,tf,'gauss',k); A_,(mx,my)=info['A'],info['mu']; U=(my+(sf.double()-mx)@A_.T).float()
    elif rule=='meanshift_full': U=sf+(tf.mean(0)-sf.mean(0))
    else: raise ValueError(rule)
    return dict(U=U,X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,info={})
def roundtrip_dist(Z,Xr,seed=0):
    """set-level round-trip: energy distance between intended cloud and re-encoded cloud, over the noise floor of two halves of Z."""
    g=torch.Generator(device=device).manual_seed(seed); p=torch.randperm(len(Z),generator=g,device=device); h=len(Z)//2
    floor=energy_dist(Z[p[:h]],Z[p[h:]],seed=seed); return energy_dist(Z,Xr,seed=seed)/(floor+1e-9)
print('helpers-2 ready')

In [ ]:
# ============================================================
# E6b. MOTIF RECOVERY WITH CONTROLS — is the 'decoder regenerates B's motifs' result real or a non-discriminative metric?
#   controls against B's clusters: recon A, unrelated CNT recon, source copy; plus within-cluster distance and energy distance to B
# ============================================================
from sklearn.cluster import KMeans
_km=globals().get('_km',{})
CSV=f'{MECH}/E6b_controls.csv'; rows,have=load_rows(CSV,['k','cond']); t0=time.time()
KC=8; all_dtd=sorted(glob.glob(f'{LOCAL}/*/*.jpg')); rng=np.random.RandomState(7)
for n_,k in enumerate(DEV):
    k=int(k); d=get_d(k); L,Cc,H,W,sf,tf=_grids(d)
    i_s=_sub(len(sf),k,'s'); i_t=_sub(len(tf),k,'t'); src,tgt=sf[i_s],tf[i_t]
    if k not in _km: _km[k]=KMeans(KC,n_init=4,random_state=0).fit(tf.detach().cpu().numpy().astype(np.float32))
    km=_km[k]; cen=torch.as_tensor(km.cluster_centers_,device=device).float()
    labB=km.predict(tf.detach().cpu().numpy().astype(np.float32)); hB=np.bincount(labB,minlength=KC)/len(labB); rare_c=int(hB.argmin())
    wc_B=float(torch.cdist(tf,cen).min(1).values.mean())                                   # B's own within-cluster spread
    ed_floor=energy_dist(tf[i_t],tf[_sub(len(tf),k+1,'t')])                                  # B vs itself (resample)
    def measures(cloud,tag):
        lab=km.predict(cloud.detach().cpu().numpy().astype(np.float32)); h=np.bincount(lab,minlength=KC)/len(lab)
        return dict(tv=float(.5*np.abs(h-hB).sum()),rare_share=float(h[rare_c]),rare_share_B=float(hB[rare_c]),
                    wc_dist=float(torch.cdist(cloud.float(),cen).min(1).values.mean()/wc_B),ed_to_B=energy_dist(cloud,tf)/(ed_floor+1e-9))
    a=anchors(k)
    ctrl={'reconB':reencode_field(a['rB']),'reconA':reencode_field(a['rA']),'source_copy_field':sf}
    f_unrel=all_dtd[rng.randint(len(all_dtd))]; ctrl['unrelated_recon']=reencode_field(to_pil(make_recon(Image.open(f_unrel).convert('RGB'))))
    for c,cloud in ctrl.items():
        if (k,c) not in have: rows.append(dict(k=k,cond=c,stage='reencoded',**measures(cloud,c)))
    for e in [0.005,0.05,0.5]:
        c=f'ot{e}'
        if (k,c) in have: continue
        P,f_,g_,_,res,it=sinkhorn_log(src,tgt,e); T=((P/P.sum(1,keepdim=True))@tgt.double()).float()
        U=T[torch.cdist(sf,src).argmin(1)]; fd=dict(X=sf,Y=tf,L=L,Cc=Cc,H=H,W=W,U=U)
        Xr=reencode_field(to_pil(decode_field(d,fd,U)))
        rows.append(dict(k=k,cond=c,stage='projected',**measures(T,c)))
        rows.append(dict(k=k,cond=c,stage='reencoded',**measures(Xr,c),rt_dist=roundtrip_dist(U,Xr)))
    for c in ['exact','uniform','gauss_full','meanshift_full']:
        if (k,c) in have: continue
        fd=affine_full(d,c,k) if c.endswith('_full') else field(d,c,k)
        Xr=reencode_field(to_pil(decode_field(d,fd,fd['U'])))
        rows.append(dict(k=k,cond=c,stage='projected',**measures(fd['U'],c)))
        rows.append(dict(k=k,cond=c,stage='reencoded',**measures(Xr,c),rt_dist=roundtrip_dist(fd['U'],Xr)))
    save_rows(rows,CSV); print(f'{n_+1}/{len(DEV)} {time.time()-t0:.0f}s')
g=pd.DataFrame(rows); g.to_csv(CSV,index=False)
print('\n=== against B\'s 8 clusters; tv/wc_dist/ed_to_B lower = closer to B (wc_dist and ed_to_B are relative to B\'s own floor) ===')
print(g.pivot_table(index='cond',columns='stage',values=['tv','wc_dist','ed_to_B','rare_share'],aggfunc='median').round(3))
print('\nrare cluster share in B (median): %.3f | rt_dist (set-level round-trip, 1 = noise floor):'%g.rare_share_B.median()); print(g[g.stage=="reencoded"].groupby('cond').rt_dist.median().round(2))
print('\nREAD: if reconA / unrelated_recon score like the ot rows, the cluster metric is non-discriminative and the "decoder regenerates B" claim is void.')
print('E6b saved')

In [ ]:
# ============================================================
# E7b. CONFIRMATION, CORRECTED — KID vs the full 2819 reference, m-out-of-n paired subsampling (no duplicates), point ΔKID reported;
#      unrelated control passed through CNT and disjoint from endpoints + reference; gauss_full / meanshift_full rows added
# ============================================================
def kid_ub(X,Y,kyy_sum=None):
    """unbiased polynomial-kernel MMD², unequal sizes, X (m,d) vs full Y (n,d)."""
    dd=X.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3; m=len(X); n=len(Y)
    kxx=kk(X,X); kxy=kk(X,Y); kyy_s=kyy_sum if kyy_sum is not None else ((kk(Y,Y).sum()-n)/(n*(n-1)))
    return float((kxx.sum()-np.trace(kxx))/(m*(m-1))+kyy_s-2*kxy.mean())
REF=ref_inc.astype(np.float64); dd=REF.shape[1]; _kyy=((REF@REF.T/dd+1)**3); KYY=(_kyy.sum()-len(REF))/(len(REF)*(len(REF)-1)); del _kyy
# --- extra method dirs on the 120 pairs ---
NEW2={'gauss_full':f'{SCALE}/gauss_full','meanshift_full':f'{SCALE}/meanshift_full','unrelated_recon':f'{SCALE}/unrelated_recon'}
for p in NEW2.values(): os.makedirs(p,exist_ok=True)
ref_names={os.path.basename(l.strip()) for l in open(f'{BASE}/pairs_120/ref_list.txt')}
end_names={os.path.basename(str(p_.get('fA',''))) for p_ in MAN_ALL}|{os.path.basename(str(p_.get('fB',''))) for p_ in MAN_ALL}
pool=[f for f in sorted(glob.glob(f'{LOCAL}/*/*.jpg')) if os.path.basename(f) not in ref_names and os.path.basename(f) not in end_names]
print('unrelated pool (disjoint from reference and all endpoints):',len(pool)); rng=np.random.RandomState(2026); t0=time.time()
for n_,k in enumerate(KS120):
    if all(os.path.exists(f'{p}/{k:04d}.png') for p in NEW2.values()): continue
    d=get_d(k)
    for c in ['gauss_full','meanshift_full']:
        fd=affine_full(d,c,k); to_pil(decode_field(d,fd,zt(fd,0.5))).save(f'{NEW2[c]}/{k:04d}.png')
    to_pil(make_recon(Image.open(pool[rng.randint(len(pool))]).convert('RGB'))).save(f'{NEW2["unrelated_recon"]}/{k:04d}.png')
    if n_%20==0: print(f'gen {n_}/120 {time.time()-t0:.0f}s')
METH={**{m:f'{SCALE}/{m}' for m in ['ot','exact','ot_sym','gaussian_w2','pixel_linear','texton_linear']},'tm':f'{SCALE}/mid',
      'proj_hard':glob.glob(f'{SCALE}/proj_hard_lam*')[0] if glob.glob(f'{SCALE}/proj_hard_lam*') else '',
      'target_mean':f'{SCALE}/target_mean','meanshift':f'{SCALE}/meanshift','source_copy':f'{SCALE}/source_copy','target_copy':f'{SCALE}/target_copy',**NEW2}
METH={m:p for m,p in METH.items() if p and all(os.path.exists(f'{p}/{k:04d}.png') for k in KS120)}
anc=dict(np.load(f'{OUT}/anchor_dino.npz')); XI={}; XD={}; rows=[]
for m,pdir in METH.items():
    fI,fD=f'{OUT}/feat_{m}_inc.npy',f'{OUT}/feat_{m}_dino.npy'; ks_all=[int(p_['k']) for p_ in MAN_ALL if os.path.exists(f'{pdir}/{int(p_["k"]):04d}.png')]
    if os.path.exists(fI) and len(ks_all)>=1000 and len(np.load(fI,mmap_mode='r'))==len(ks_all):
        idx=[i for i,k in enumerate(ks_all) if k in KSET]; XI[m]=np.load(fI)[idx]; XD[m]=np.load(fD)[idx]; ks=[ks_all[i] for i in idx]
    else:
        ims=[at128(Image.open(f'{pdir}/{k:04d}.png')) for k in KS120]; XI[m]=inc_feats(ims); XD[m]=dino_feats(ims); ks=KS120[:]
    order=np.argsort(ks); XI[m]=XI[m][order].astype(np.float64); XD[m]=XD[m][order]; ks=sorted(ks)
    rI=realism(XI[m].astype(np.float32),MANI_INC); rD=realism(XD[m],MANI_DINO)
    for i,k in enumerate(ks):
        dA=np.linalg.norm(XD[m][i]-anc[f'{k}_A']); dB=np.linalg.norm(XD[m][i]-anc[f'{k}_B']); rows.append(dict(k=k,method=m,real_inc=rI[i],real_dino=rD[i],place=dA/(dA+dB+1e-9)))
sc=pd.DataFrame(rows); sc.to_csv(f'{MECH}/E7b_scores.csv',index=False); piv={c:sc.pivot(index='k',columns='method',values=c) for c in ['real_inc','real_dino','place']}
pt={m:kid_ub(XI[m],REF,KYY) for m in METH}
B=300; msub=96; rngb=np.random.RandomState(0); draws={m:[] for m in METH}
for b in range(B):
    idx=rngb.choice(120,msub,replace=False)                                   # m-out-of-n WITHOUT replacement: no duplicate bias
    for m in METH: draws[m].append(kid_ub(XI[m][idx],REF,KYY))
draws={m:np.array(v) for m,v in draws.items()}
print(f"\n=== E7b, 120 pairs, KID vs full reference (n=2819); ΔKID = point difference, CI from shared m=96-of-120 subsamples ===")
print(f"{'method':16}{'KID':>8}{'dKID vs ot':>12}{'95% CI (paired)':>20}{'realInc':>9}{'realDINO':>10}{'|pl-.5|':>9}{'dRealDINO vs ot [CI]':>24}")
res={}
for m in sorted(METH,key=lambda m:pt[m]):
    dpt=pt[m]-pt['ot']; dd_=draws[m]-draws['ot']; lo,hi=np.percentile(dd_,[2.5,97.5])
    dr=(piv['real_dino'][m]-piv['real_dino']['ot']).dropna().values; rlo,rhi=boot_ci(dr) if m!='ot' else (0,0)
    res[m]=dict(KID=pt[m],dKID=dpt,dKID_ci=[lo,hi],RInc=float(piv['real_inc'][m].median()),RDin=float(piv['real_dino'][m].median()),Place=float((piv['place'][m]-.5).abs().median()))
    print(f"{m:16}{pt[m]:8.4f}{dpt:+12.4f}   [{lo:+.4f},{hi:+.4f}]{res[m]['RInc']:9.3f}{res[m]['RDin']:10.3f}{res[m]['Place']:9.3f}   {np.median(dr):+.3f} [{rlo:+.3f},{rhi:+.3f}]")
json.dump(res,open(f'{MECH}/E7b_numbers.json','w'),indent=1,default=float); print('E7b saved')

In [ ]:
# ============================================================
# N1000. TARGET-MEAN and GAUSS-FULL at n=1000 — decode all pairs (resumable), score with the canonical 128px protocol,
#        print next to numbers_final128.json for the paper's rules table
# ============================================================
NEWK={'target_mean':f'{SCALE}/target_mean','gauss_full':f'{SCALE}/gauss_full'}
for p in NEWK.values(): os.makedirs(p,exist_ok=True)
t0=time.time(); todo=[int(p_['k']) for p_ in MAN_ALL if not all(os.path.exists(f'{p}/{int(p_["k"]):04d}.png') for p in NEWK.values())]
print(len(todo),'pairs to decode')
for n_,k in enumerate(todo):
    d=get_d(k)
    if not os.path.exists(f'{NEWK["target_mean"]}/{k:04d}.png'):
        fd=field(d,'uniform',k); to_pil(decode_field(d,fd,zt(fd,0.5))).save(f'{NEWK["target_mean"]}/{k:04d}.png')
    if not os.path.exists(f'{NEWK["gauss_full"]}/{k:04d}.png'):
        fd=affine_full(d,'gauss_full',k); to_pil(decode_field(d,fd,zt(fd,0.5))).save(f'{NEWK["gauss_full"]}/{k:04d}.png')
    if n_%100==0 and n_: el=time.time()-t0; print(f'{n_}/{len(todo)} {el:.0f}s ETA {(len(todo)-n_)*el/n_/60:.0f} min')
anc=dict(np.load(f'{OUT}/anchor_dino.npz')); base=json.load(open(f'{OUT}/numbers_final128.json'))
print(f"\n=== CANONICAL 128px, n=1000 ===\n{'method':14}{'FID':>8}{'KID':>16}{'realInc':>9}{'realDINO':>10}{'|place-.5|':>11}")
for m,v in base.items(): print(f"{m:14}{v['FID']:8.1f}{v['KID']:9.4f}         {v['RInc']:9.3f}{v['RDin']:10.3f}{v['Place']:11.3f}   (existing)")
out={}
for m,pdir in NEWK.items():
    fI,fD=f'{OUT}/feat_{m}_inc.npy',f'{OUT}/feat_{m}_dino.npy'
    if os.path.exists(fI) and len(np.load(fI,mmap_mode='r'))==len(MAN_ALL): XI=np.load(fI); XD=np.load(fD)
    else:
        ims=[at128(Image.open(f'{pdir}/{int(p_["k"]):04d}.png')) for p_ in MAN_ALL]; XI=inc_feats(ims); XD=dino_feats(ims); np.save(fI,XI); np.save(fD,XD)
    ks=[int(p_['k']) for p_ in MAN_ALL]; rI=realism(XI,MANI_INC); rD=realism(XD,MANI_DINO)
    plc=np.array([np.linalg.norm(XD[i]-anc[f'{k}_A'])/(np.linalg.norm(XD[i]-anc[f'{k}_A'])+np.linalg.norm(XD[i]-anc[f'{k}_B'])+1e-9) for i,k in enumerate(ks)])
    Fv=fid(XI,ref_inc); Kv=kid(XI,ref_inc)
    out[m]=dict(FID=round(Fv,1),KID=round(Kv[0],4),KIDsd=round(Kv[1],4),RInc=round(float(np.median(rI)),3),RDin=round(float(np.median(rD)),3),Place=round(float(np.median(np.abs(plc-.5))),3))
    print(f"{m:14}{Fv:8.1f}{Kv[0]:9.4f}±{Kv[1]:.4f}{out[m]['RInc']:9.3f}{out[m]['RDin']:10.3f}{out[m]['Place']:11.3f}   (NEW)")
json.dump(out,open(f'{OUT}/numbers_final128_extra.json','w'),indent=1); print('saved numbers_final128_extra.json')

In [ ]:
from scipy import linalg
def fid(X,Y):
    mx,my=X.mean(0),Y.mean(0); Sx,Sy=np.cov(X,rowvar=False),np.cov(Y,rowvar=False)
    cm,_=linalg.sqrtm(Sx@Sy,disp=False); cm=cm.real if np.iscomplexobj(cm) else cm
    return float(((mx-my)**2).sum()+np.trace(Sx+Sy-2*cm))

In [ ]:
# ============================================================
# E8. DECODER-INDUCED METRIC — (a) central-difference response along sharp−smooth vs phase-randomized controls that keep
#     each channel's amplitude spectrum AND all cross-spectra (channel covariance); (b) all 2^4 sharp/smooth stage
#     combinations with main effects + interactions; (c) Euclidean vs perceptual vs decoder-induced distance at t=.5 and .9
# ============================================================
CSV=f'{MECH}/E8_metric.csv'; rows,have=load_rows(CSV,['k','t','kind','name','h']); t0=time.time()
def phase_randomize(D_,H,W,seed):
    """same per-channel amplitude spectrum and same cross-spectra as D_ (N,d): one shared random phase field across channels."""
    G=D_.reshape(H,W,-1).permute(2,0,1).double(); Fh=torch.fft.rfft2(G)
    g=torch.Generator(device=D_.device).manual_seed(seed); phi=torch.rand(Fh.shape[-2:],generator=g,device=D_.device,dtype=torch.float64)*2*math.pi
    R=torch.fft.irfft2(Fh*torch.exp(1j*phi)[None],s=(H,W)); return R.permute(1,2,0).reshape(H*W,-1)
def pix(p): return np.asarray(p,dtype=np.float32)/255.
HS=[0.005,0.01,0.02]; NSTAGE=len(model.generator.spade_blocks)
for n_,k in enumerate(DEV[:12]):
    k=int(k); d=get_d(k); Wm,Fm=splat_W(d,'a'); Q,_=projector(Wm); fo=field(d,'ot',k); fh=field(d,'exact',k)
    X=fo['X'].double(); Uo=fo['U'].double(); Uh=fh['U'].double(); H,W_,Cc,L=fo['H'],fo['W'],fo['Cc'],fo['L']
    for t in [0.5,0.9]:
        Zs=(1-t)*X+t*Uo; Zh=(1-t)*X+t*Uh; Delta=Zh-Zs; nZ=float(Zs.norm()); nD=float(Delta.norm())
        # ---- (c) three distances between the sharp and smooth fields at this t ----
        if (k,t,'dist','sharp_vs_smooth',0) not in have:
            ps=to_pil(decode_field(d,fo,Zs)); ph=to_pil(decode_field(d,fo,Zh)); fe=dino_feats([at128(ps),at128(ph)])
            rows.append(dict(k=k,t=t,kind='dist',name='sharp_vs_smooth',h=0,eucl_rel=nD/nZ,lpips=lpips_d(ps,ph),dino=float(np.linalg.norm(fe[0]-fe[1])),pix_rms=float(np.sqrt(((pix(ps)-pix(ph))**2).mean()))))
        # ---- (a) central-difference response along matched directions ----
        g=torch.Generator(device=device).manual_seed(k+31); white=torch.randn_like(Delta,generator=g)
        dirs={'sharp_minus_smooth':Delta}
        for s in range(3): dirs[f'phaseRand{s}']=phase_randomize(Delta,H,W_,seed=k*10+s)      # spectrum + channel-covariance matched
        dirs['white_random']=white; dirs['inSub_random']=proj(Q,white); dirs['amplitude']=Zs-Zs.mean(0)
        base=to_pil(decode_field(d,fo,Zs)); fb=dino_feats([at128(base)])[0]
        for name,v in dirs.items():
            v=v/v.norm()
            for h in HS:
                if (k,t,'resp',name,h) in have: continue
                pp=to_pil(decode_field(d,fo,Zs+h*nZ*v)); pm=to_pil(decode_field(d,fo,Zs-h*nZ*v)); fe=dino_feats([at128(pp),at128(pm)])
                rows.append(dict(k=k,t=t,kind='resp',name=name,h=h,resp_pix=float(np.sqrt(((pix(pp)-pix(pm))**2).mean()))/(2*h),
                                 resp_lpips=lpips_d(pp,pm)/(2*h),resp_dino=float(np.linalg.norm(fe[0]-fe[1]))/(2*h)))
        # ---- (b) all 2^4 stage combinations: hard at stages in S, smooth elsewhere ----
        sm_s=sm_from_field(Zs.float(),L,H,W_,Cc); sm_h=sm_from_field(Zh.float(),L,H,W_,Cc)
        all_s=to_pil(decode_stagewise(d['blob_a'],[sm_s]*NSTAGE)); all_h=to_pil(decode_stagewise(d['blob_a'],[sm_h]*NSTAGE)); fe=dino_feats([at128(all_s),at128(all_h)])
        for S in itertools.product([0,1],repeat=NSTAGE):
            name='S'+''.join(map(str,S))
            if (k,t,'combo',name,0) in have: continue
            pil=to_pil(decode_stagewise(d['blob_a'],[sm_h if S[j] else sm_s for j in range(NSTAGE)])); f=dino_feats([at128(pil)])[0]
            rows.append(dict(k=k,t=t,kind='combo',name=name,h=0,**{f's{j}':S[j] for j in range(NSTAGE)},lpips_vs_smooth=lpips_d(pil,all_s),lpips_vs_sharp=lpips_d(pil,all_h),
                             dino_vs_smooth=float(np.linalg.norm(f-fe[0])),dino_vs_sharp=float(np.linalg.norm(f-fe[1]))))
    save_rows(rows,CSV); print(f'{n_+1}/12 {time.time()-t0:.0f}s')
g8=pd.DataFrame(rows); g8.to_csv(CSV,index=False)
print('\n(a) central-difference response per unit relative perturbation; the claim needs sharp_minus_smooth << phaseRand* at matched spectrum/covariance:')
print(g8[g8.kind=='resp'].pivot_table(index='name',columns=['t','h'],values='resp_lpips',aggfunc='median').round(2))
print('\n(c) sharp vs smooth: Euclidean feature distance (relative) vs perceptual (LPIPS) vs DINO, by t — prediction: perceptual/Euclidean ratio grows with t:')
dc=g8[g8.kind=='dist'].groupby('t')[['eucl_rel','lpips','dino']].median(); dc['lpips_per_eucl']=dc.lpips/dc.eucl_rel; print(dc.round(3))
print('\n(b) stage main effects + pairwise interactions on LPIPS-vs-smooth (linear model over the 16 combos, per t, median over pairs):')
import numpy.linalg as npl
for t in [0.5,0.9]:
    coefs=[]
    for k_,s in g8[(g8.kind=='combo')&(g8.t==t)].groupby('k'):
        S=s[[f's{j}' for j in range(NSTAGE)]].values.astype(float); y=s.lpips_vs_smooth.values
        cols=[np.ones(len(S))]+[S[:,j] for j in range(NSTAGE)]+[S[:,i]*S[:,j] for i in range(NSTAGE) for j in range(i+1,NSTAGE)]
        A=np.stack(cols,1); coefs.append(npl.lstsq(A,y,rcond=None)[0])
    names=['const']+[f'stage{j}' for j in range(NSTAGE)]+[f's{i}xs{j}' for i in range(NSTAGE) for j in range(i+1,NSTAGE)]
    print(f'  t={t}: '+'  '.join(f'{n}={c:+.3f}' for n,c in zip(names,np.median(np.stack(coefs),0))))
print('E8 saved')

In [ ]:
# ============================================================
# N1000-UNC. PAIRED UNCERTAINTY AT n=1000 — KID: delete-one-pair jackknife (exact for a U-statistic);
#            FID: paired bootstrap over pairs with shared draws (GPU eigvalsh). Contrasts the review asked for.
# ============================================================
METHS=['ot','target_mean','gauss_full','exact','ot_sym','pixel_linear','tm']
ks_ref=[int(p_['k']) for p_ in MAN_ALL]
XI={}
for m in METHS:
    pdir=f'{SCALE}/mid' if m=='tm' else f'{SCALE}/{m}'
    ks_m=[int(p_['k']) for p_ in MAN_ALL if os.path.exists(f'{pdir}/{int(p_["k"]):04d}.png')]
    X=np.load(f'{OUT}/feat_{m}_inc.npy'); assert len(X)==len(ks_m)==1000, (m,len(X),len(ks_m))
    XI[m]=X[np.argsort(ks_m)].astype(np.float64)          # align all methods to sorted k
REF=ref_inc.astype(np.float64); dd=REF.shape[1]; kk=lambda a,b:(a@b.T/dd+1)**3
KYY=kk(REF,REF); KYY=(KYY.sum()-len(REF))/(len(REF)*(len(REF)-1))
def kid_loo(X):
    """returns (point KID vs full reference, vector of delete-one KIDs) — exact leave-one-out of the U-statistic."""
    m=len(X); kxx=kk(X,X); np.fill_diagonal(kxx,0); S=kxx.sum(1); T=S.sum(); r=kk(X,REF).mean(1); R=r.sum()
    point=T/(m*(m-1))+KYY-2*R/m
    loo=(T-2*S)/((m-1)*(m-2))+KYY-2*(R-r)/(m-1)
    return point,loo
K={m:kid_loo(XI[m]) for m in METHS}
def jk_ci(dloo,n):
    """jackknife SE of a paired difference from its delete-one values; normal 95% CI."""
    se=np.sqrt((n-1)/n*((dloo-dloo.mean())**2).sum()); return se
# --- FID with paired pair-resampling, GPU ---
Rt=torch.tensor(REF,device=device); mR=Rt.mean(0); SR=torch.cov(Rt.T); eR,vR=torch.linalg.eigh(SR); SRh=(vR*eR.clamp_min(0).sqrt())@vR.T
def fid_gpu(X):
    Xt=torch.tensor(X,device=device); m=Xt.mean(0); S=torch.cov(Xt.T); lam=torch.linalg.eigvalsh(SRh@S@SRh).clamp_min(0)
    return float(((m-mR)**2).sum()+torch.trace(S)+torch.trace(SR)-2*lam.sqrt().sum())
Bn=200; rng=np.random.RandomState(0); fboot={m:[] for m in METHS}; fpt={m:fid_gpu(XI[m]) for m in METHS}; t0=time.time()
for b in range(Bn):
    idx=rng.choice(1000,1000,replace=True)
    for m in METHS: fboot[m].append(fid_gpu(XI[m][idx]))
    if b%50==0: print(f'fid boot {b}/{Bn} {time.time()-t0:.0f}s')
fboot={m:np.array(v) for m,v in fboot.items()}
print(f"\n{'method':13}{'FID':>7}{'FID 95% CI':>18}{'KID':>9}{'KID jk-SE':>11}")
for m in METHS:
    lo,hi=np.percentile(fboot[m],[2.5,97.5]); print(f"{m:13}{fpt[m]:7.1f}   [{lo:6.1f},{hi:6.1f}]{K[m][0]:9.4f}{jk_ci(K[m][1],1000):11.4f}")
print(f"\n=== paired contrasts, n=1000 (ΔFID: point, bootstrap 95% CI shared draws; ΔKID: point, jackknife 95% CI) ===")
print(f"{'contrast':26}{'dFID':>8}{'95% CI':>18}{'dKID':>9}{'95% CI':>20}")
res={}
for a,b in [('ot','target_mean'),('ot','gauss_full'),('exact','gauss_full'),('ot_sym','gauss_full'),('ot','pixel_linear'),('ot','exact'),('tm','ot')]:
    dF=fpt[a]-fpt[b]; dfb=fboot[a]-fboot[b]; flo,fhi=np.percentile(dfb,[2.5,97.5])
    dK=K[a][0]-K[b][0]; se=jk_ci(K[a][1]-K[b][1],1000); klo,khi=dK-1.96*se,dK+1.96*se
    res[f'{a}-{b}']=dict(dFID=dF,dFID_ci=[flo,fhi],dKID=dK,dKID_ci=[klo,khi])
    print(f"{a+' - '+b:26}{dF:+8.1f}   [{flo:+6.1f},{fhi:+6.1f}]{dK:+9.4f}   [{klo:+.4f},{khi:+.4f}]")
json.dump(res,open(f'{MECH}/n1000_paired_uncertainty.json','w'),indent=1,default=float)
print("\nREAD: a CI containing 0 = 'numerically matched, not distinguishable at n=1000'; state an equivalence margin before calling anything 'equal'.")

In [ ]:
# ============================================================
# RETRIEVAL. Decode A_i's layout with the target-mean field of B_j (and the diffuse-OT field toward B_j) for all i,j in DEV;
#            re-encode; rank the true B_j among all DEV targets by raw energy distance. Chance top-1 = 1/24.
# ============================================================
KD=[int(k) for k in DEV]; nD=len(KD)
Bcloud={}; Acloud={}; Bmu={}; DD={}
for k in KD:
    d=get_d(k); DD[k]=d; L,Cc,H,W,sf,tf=_grids(d); Acloud[k]=sf; Bcloud[k]=tf; Bmu[k]=tf.mean(0)
def ed_raw(X,Y,m=2048,seed=0):
    g=torch.Generator(device=device).manual_seed(seed); x=X[torch.randperm(len(X),generator=g,device=device)[:m]].double(); y=Y[torch.randperm(len(Y),generator=g,device=device)[:m]].double()
    return float(2*torch.cdist(x,y).mean()-torch.cdist(x,x).mean()-torch.cdist(y,y).mean())
CSV=f'{MECH}/retrieval.csv'; rows,have=load_rows(CSV,['ki','kj','rule']); t0=time.time()
for a_,ki in enumerate(KD):
    d=DD[ki]; L,Cc,H,W,sf,_=_grids(d)
    for kj in KD:
        for rule in ['target_mean','ot0.05']:
            if (ki,kj,rule) in have: continue
            if rule=='target_mean': U=Bmu[kj][None].expand_as(sf).clone()
            else:
                i_s=_sub(len(sf),ki,'s'); i_t=_sub(len(Bcloud[kj]),kj,'t'); P,_=sinkhorn_plan(sf[i_s],Bcloud[kj][i_t],EPS)
                tt=(P/(P.sum(1,keepdim=True)+1e-30))@Bcloud[kj][i_t]; U=tt[torch.cdist(sf,sf[i_s]).argmin(1)]
            fd=dict(X=sf,Y=Bcloud[kj],L=L,Cc=Cc,H=H,W=W,U=U); Xr=reencode_field(to_pil(decode_field(d,fd,U)))   # eta=1
            dists={kb:ed_raw(Xr,Bcloud[kb]) for kb in KD}; order=sorted(KD,key=lambda kb:dists[kb])
            rows.append(dict(ki=ki,kj=kj,rule=rule,rank_true=order.index(kj)+1,top1=int(order[0]==kj),
                             ed_true=dists[kj],ed_best_other=min(v for kb,v in dists.items() if kb!=kj),ed_to_A=ed_raw(Xr,Acloud[ki])))
    save_rows(rows,CSV); print(f'{a_+1}/{nD} {time.time()-t0:.0f}s')
g=pd.DataFrame(rows); g.to_csv(CSV,index=False)
print(f"\n=== target retrieval after decode+re-encode (chance top-1 = {1/nD:.3f}, chance mean rank = {(nD+1)/2:.1f}) ===")
print(g.groupby('rule').agg(top1=('top1','mean'),mean_rank=('rank_true','mean'),median_rank=('rank_true','median'),
                            frac_true_beats_all=('rank_true',lambda r:(r==1).mean())).round(3))
own=g[g.ki==g.kj].groupby('rule')[['ed_true','ed_to_A']].median().round(3); print('\nown-pair (i=j): ED to true B vs ED to A after decode:'); print(own)
print('\nREAD: top-1 far above chance = decoder moves output toward its SPECIFIC target (target-specific regularization);')
print('      near chance = decoder only contracts toward a generic manifold and the ED reduction is not target-specific.')

In [ ]:
# ============================================================
# E8b. (i) full 2^4 factorial, effect coding (all 15 effects exactly identified per pair); (ii) mixed-stage combos far from BOTH
#      all-smooth and all-sharp?; (iii) paired CI of sharp/phaseRand response ratio; (iv) phase control re-implemented with
#      full FFT (Hermitian by construction) and verified: amplitude spectra + cross-spectra preserved
# ============================================================
g8=pd.read_csv(f'{MECH}/E8_metric.csv'); NS=4
# (i) effect coding: x_j in {-1,+1}; all interactions; per pair, then median + bootstrap CI over pairs
def design(S):  # S: (16,4) in {0,1}
    x=2*S-1; cols=[np.ones(len(S))]; names=['mean']
    for r in range(1,NS+1):
        for c in itertools.combinations(range(NS),r): cols.append(np.prod(x[:,list(c)],1)); names.append('x'.join(f's{j}' for j in c))
    return np.stack(cols,1),names
for t in [0.5,0.9]:
    eff=[]
    for k_,s in g8[(g8.kind=='combo')&(g8.t==t)].groupby('k'):
        s=s.sort_values('name'); A,names=design(s[[f's{j}' for j in range(NS)]].values.astype(float)); eff.append(np.linalg.solve(A,s.lpips_vs_smooth.values))
    E=np.stack(eff); med=np.median(E,0); ci=np.array([boot_ci(E[:,i]) for i in range(E.shape[1])])
    print(f'\n(i) t={t}: effect-coded factorial on LPIPS-vs-smooth (median over {len(E)} pairs [95% CI]); |effect| = half the change when that stage/interaction flips')
    for n_,m_,(lo,hi) in sorted(zip(names,med,ci),key=lambda z:-abs(z[1]))[:8]: print(f'   {n_:16}{m_:+.3f} [{lo:+.3f},{hi:+.3f}]')
# (ii) third texture: mixed combos vs the sharp-vs-smooth distance itself
for t in [0.5,0.9]:
    c=g8[(g8.kind=='combo')&(g8.t==t)]; ref=g8[(g8.kind=='dist')&(g8.t==t)].set_index('k').lpips
    c=c.assign(sharp_smooth=c.k.map(ref)); c['far_from_both']=(c.lpips_vs_smooth>c.sharp_smooth)&(c.lpips_vs_sharp>c.sharp_smooth)
    print(f'\n(ii) t={t}: combos farther from BOTH endpoints than the endpoints are from each other (median over pairs):')
    print(c.groupby('name')[['lpips_vs_smooth','lpips_vs_sharp','far_from_both']].median().round(3).sort_values('far_from_both',ascending=False).head(8))
# (iii) paired ratio sharp / mean(phaseRand), per pair and h
r=g8[g8.kind=='resp']; pr=r[r.name.str.startswith('phaseRand')].groupby(['k','t','h']).resp_lpips.mean().rename('phase')
sh=r[r.name=='sharp_minus_smooth'].set_index(['k','t','h']).resp_lpips.rename('sharp'); q=pd.concat([sh,pr],axis=1).dropna(); q['ratio']=q.sharp/q.phase
print('\n(iii) response ratio sharp_minus_smooth / phaseRand (LPIPS), median [95% CI over pairs]; 1 = ordinary direction:')
for (t,h),s in q.groupby(level=['t','h']):
    lo,hi=boot_ci(s.ratio.values); print(f'   t={t} h={h}: {s.ratio.median():.3f} [{lo:.3f},{hi:.3f}]  n={len(s)}')
# (iv) corrected phase control: full fft2 with phase from a real random field (Hermitian by construction); verify
def phase_randomize_v2(D_,H,W,seed):
    G=D_.reshape(H,W,-1).permute(2,0,1).double(); Fh=torch.fft.fft2(G)
    g=torch.Generator(device=D_.device).manual_seed(seed); noise=torch.randn(H,W,generator=g,device=D_.device,dtype=torch.float64)
    phi=torch.angle(torch.fft.fft2(noise))                                      # real field -> conjugate-symmetric phase
    R=torch.fft.ifft2(Fh.abs()*torch.exp(1j*phi)[None]).real; return R.permute(1,2,0).reshape(H*W,-1)
k=int(DEV[0]); d=get_d(k); fo=field(d,'ot',k); fh=field(d,'exact',k); D_=(fh['U']-fo['U']).double(); H,W_=fo['H'],fo['W']
R_=phase_randomize_v2(D_,H,W_,seed=1)
FA=torch.fft.fft2(D_.reshape(H,W_,-1).permute(2,0,1)); FB=torch.fft.fft2(R_.reshape(H,W_,-1).permute(2,0,1))
amp_err=float((FA.abs()-FB.abs()).norm()/FA.abs().norm()); cross_err=float(((FA[:,None]*FA[None].conj()).abs()-(FB[:,None]*FB[None].conj()).abs()).norm()/(FA[:,None]*FA[None].conj()).abs().norm()) if D_.shape[1]<=64 else float('nan')
covA=torch.cov(D_.T); covB=torch.cov(R_.T); cov_err=float((covA-covB).norm()/covA.norm()); imag=0.0
print(f'\n(iv) corrected phase control on pair {k}: amplitude-spectrum rel. error {amp_err:.2e} | channel-covariance rel. error {cov_err:.2e} | norm ratio {float(R_.norm()/D_.norm()):.4f}')
print('     (cross-spectra are preserved by construction when all channels share one Hermitian phase; covariance error above is the numeric check)')

In [ ]:
# ============================================================
# E8c. response along sharp−smooth vs a VERIFIED control: shared random all-pass phase ADDED to the original spectra
#      (preserves per-channel amplitude spectra AND all cross-spectra ⇒ channel covariance), central differences
# ============================================================
def phase_shift_control(D_,H,W,seed):
    G=D_.reshape(H,W,-1).permute(2,0,1).double(); Fh=torch.fft.fft2(G)
    g=torch.Generator(device=D_.device).manual_seed(seed); noise=torch.randn(H,W,generator=g,device=D_.device,dtype=torch.float64)
    phi=torch.angle(torch.fft.fft2(noise)); R=torch.fft.ifft2(Fh*torch.exp(1j*phi)[None]).real
    return R.permute(1,2,0).reshape(H*W,-1)
# verify on one pair
k=int(DEV[0]); d=get_d(k); fo=field(d,'ot',k); fh=field(d,'exact',k); D_=(fh['U']-fo['U']).double(); H,W_=fo['H'],fo['W']
R_=phase_shift_control(D_,H,W_,1)
FA=torch.fft.fft2(D_.reshape(H,W_,-1).permute(2,0,1)); FB=torch.fft.fft2(R_.reshape(H,W_,-1).permute(2,0,1))
print(f'amp-spectrum err {float((FA.abs()-FB.abs()).norm()/FA.abs().norm()):.2e} | channel-cov err {float((torch.cov(D_.T)-torch.cov(R_.T)).norm()/torch.cov(D_.T).norm()):.2e} | spatial corr with original {float((D_*R_).sum()/(D_.norm()*R_.norm())):.3f} (should be ~0)')
CSV=f'{MECH}/E8c_control.csv'; rows,have=load_rows(CSV,['k','t','name','h']); t0=time.time()
for n_,k in enumerate(DEV[:12]):
    k=int(k); d=get_d(k); fo=field(d,'ot',k); fh=field(d,'exact',k); X=fo['X'].double(); Uo=fo['U'].double(); Uh=fh['U'].double(); H,W_=fo['H'],fo['W']
    for t in [0.5,0.9]:
        Zs=(1-t)*X+t*Uo; Delta=(Uh-Uo)*t; nZ=float(Zs.norm())
        dirs={'sharp_minus_smooth':Delta,**{f'shift{s}':phase_shift_control(Delta,H,W_,k*10+s) for s in range(3)}}
        for name,v in dirs.items():
            v=v/v.norm()
            for h in [0.005,0.01,0.02]:
                if (k,t,name,h) in have: continue
                pp=to_pil(decode_field(d,fo,Zs+h*nZ*v)); pm=to_pil(decode_field(d,fo,Zs-h*nZ*v))
                rows.append(dict(k=k,t=t,name=name,h=h,resp_lpips=lpips_d(pp,pm)/(2*h)))
    save_rows(rows,CSV); print(f'{n_+1}/12 {time.time()-t0:.0f}s')
g=pd.DataFrame(rows); g.to_csv(CSV,index=False)
ctrl=g[g.name.str.startswith('shift')].groupby(['k','t','h']).resp_lpips.mean(); sh=g[g.name=='sharp_minus_smooth'].set_index(['k','t','h']).resp_lpips
q=(sh/ctrl).dropna().rename('ratio').reset_index()
print('\nsharp_minus_smooth / covariance-preserving control (LPIPS response), median [95% CI]:')
for (t,h),s in q.groupby(['t','h']): lo,hi=boot_ci(s.ratio.values); print(f'   t={t} h={h}: {s.ratio.median():.3f} [{lo:.3f},{hi:.3f}]')